[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/05_determinants_trace_and_matrix_polynomials/exercises.ipynb)

# Module 05 — Exercises: Determinants, Trace, and Matrix Polynomials

Fifty-four solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): transpose is $A^{\top}$, norms are written
$\lVert x \rVert$, and the characteristic polynomial is the monic $p_A(t) = \det(tI - A)$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import itertools
from math import prod

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Trace of the identity

**Statement.** Compute $\operatorname{tr}(I_n)$.

**Intuition.** The trace adds up the diagonal, and every diagonal entry of $I_n$ is $1$.

**Solution.**

*Step 1.* $(I_n)_{ii} = 1$ for $i = 1, \dots, n$ by Definition 3.1.

*Step 2.* Summing gives $\sum_{i=1}^{n} 1 = n$.

$$
\boxed{\operatorname{tr}(I_n) = n}
$$

**Key takeaway.** The trace of the identity records the dimension of the space, which is why
$\operatorname{tr}(P) = \operatorname{rank}(P)$ for a projection.

In [2]:
for n in (1, 3, 7):
    print(f"n = {n}:  tr(I_n) = {np.trace(np.eye(n)):.0f}")
    assert np.trace(np.eye(n)) == n

n = 1:  tr(I_n) = 1
n = 3:  tr(I_n) = 3
n = 7:  tr(I_n) = 7


### Problem L0.2 — Determinant of a diagonal matrix

**Statement.** Let $D = \operatorname{diag}(d_1, \dots, d_n)$. Find $\det D$.

**Intuition.** A diagonal matrix stretches each axis independently, so the volume factor is the
product of the stretches.

**Solution.**

*Step 1.* In the Leibniz formula of Theorem 4.1 a term is non-zero only if every factor
$D_{\sigma(j), j}$ is non-zero, which forces $\sigma(j) = j$.

*Step 2.* Only $\sigma = \operatorname{id}$ survives, with sign $+1$.

$$
\boxed{\det D = \prod_{i=1}^{n} d_i}
$$

**Key takeaway.** The same argument works for any triangular matrix, because only the identity
permutation can avoid the zero triangle.

In [3]:
d = np.array([4.0, -1.0, 2.5, 3.0])
D = np.diag(d)
T = np.triu(rng.standard_normal((4, 4)))
print("diagonal :", np.linalg.det(D), " product of entries :", np.prod(d))
print("triangular:", np.linalg.det(T), " product of diagonal:", np.prod(np.diag(T)))
assert abs(np.linalg.det(D) - np.prod(d)) < 1e-12
assert abs(np.linalg.det(T) - np.prod(np.diag(T))) < 1e-12

diagonal : -30.000000000000014  product of entries : -30.0
triangular: 0.02074966874626761  product of diagonal: 0.0207496687462676


### Problem L0.3 — Rank of an outer product

**Statement.** For non-zero $u \in \mathbb{R}^{m}$ and $v \in \mathbb{R}^{n}$, find
$\operatorname{rank}(u v^{\top})$.

**Intuition.** Every column of $uv^{\top}$ is a multiple of the single vector $u$.

**Solution.**

*Step 1.* The $j$-th column of $u v^{\top}$ is $v_j u$.

*Step 2.* The column space is therefore contained in $\operatorname{span}(u)$, and it equals
$\operatorname{span}(u)$ because some $v_j \neq 0$.

$$
\boxed{\operatorname{rank}(u v^{\top}) = 1}
$$

**Key takeaway.** Rank-one matrices are exactly the non-zero outer products, and for $m = n \ge 2$
they are singular, so $\det(uv^{\top}) = 0$.

In [4]:
u = np.array([1.0, -2.0, 3.0])
v = np.array([4.0, 0.0, -1.0])
Ouv = np.outer(u, v)
print("outer product:\n", Ouv)
print("rank:", np.linalg.matrix_rank(Ouv), "  det:", np.linalg.det(Ouv))
assert np.linalg.matrix_rank(Ouv) == 1
assert abs(np.linalg.det(Ouv)) < 1e-12

outer product:
 [[ 4.  0. -1.]
 [-8. -0.  2.]
 [12.  0. -3.]]
rank: 1   det: 0.0


### Problem L0.4 — Hadamard product with the all-ones matrix

**Statement.** Let $J \in \mathbb{R}^{m \times n}$ have every entry equal to $1$. Compute
$A \circ J$.

**Intuition.** Multiplying entrywise by $1$ changes nothing.

**Solution.**

*Step 1.* By Definition 3.7, $(A \circ J)_{ij} = A_{ij} J_{ij} = A_{ij} \cdot 1$.

$$
\boxed{A \circ J = A}
$$

**Key takeaway.** $J$ is the identity element for $\circ$, not $I$. The Hadamard product has its
own algebra, which is why $\det$ and $\operatorname{tr}$ behave so differently under it.

In [5]:
A = rng.standard_normal((3, 4))
J = np.ones((3, 4))
print("||A o J - A||_F =", np.linalg.norm(A * J - A))
print("A o I is the diagonal part:\n", np.round(np.eye(3) * A[:, :3], 4))
assert np.allclose(A * J, A)

||A o J - A||_F = 0.0
A o I is the diagonal part:
 [[-0.5443 -0.      0.    ]
 [-0.      1.3665 -0.    ]
 [ 0.      0.     -0.7435]]


### Problem L0.5 — The $2 \times 2$ determinant

**Statement.** Compute $\det \begin{pmatrix} a & b \\ c & d \end{pmatrix}$ from the Leibniz
formula.

**Intuition.** $S_2$ has two permutations, so the sum has two terms of opposite sign.

**Solution.**

*Step 1.* The identity permutation contributes $+ a d$.

*Step 2.* The transposition contributes $- c b$.

$$
\boxed{\det \begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc}
$$

**Key takeaway.** This is the signed area of the parallelogram with edges $(a,c)$ and $(b,d)$,
the picture drawn in Section 2 of the theory notebook.

In [6]:
a, b, c, d = 3.0, 1.0, 4.0, 2.0
M = np.array([[a, b], [c, d]])
print("ad - bc =", a * d - b * c, "   numpy:", np.linalg.det(M))
assert abs(np.linalg.det(M) - (a * d - b * c)) < 1e-12

ad - bc = 2.0    numpy: 2.0


### Problem L0.6 — Determinant of a product and an inverse

**Statement.** Given $\det A = 3$ and $\det B = 5$ for square matrices of the same size, find
$\det(A^{2}B^{-1})$.

**Intuition.** The determinant turns matrix products into scalar products.

**Solution.**

*Step 1.* By Theorem 4.3, $\det(A^{2}B^{-1}) = \det(A)^{2}\det(B^{-1})$.

*Step 2.* Also by Theorem 4.3, $\det(B^{-1}) = 1/\det(B)$.

$$
\boxed{\det(A^{2}B^{-1}) = \frac{3^{2}}{5} = \frac{9}{5}}
$$

**Key takeaway.** Volume factors compose multiplicatively, and inversion inverts them.

In [7]:
A = np.diag([3.0, 1.0, 1.0])
B = np.diag([5.0, 1.0, 1.0])
val = np.linalg.det(A @ A @ np.linalg.inv(B))
print("det A =", np.linalg.det(A), " det B =", np.linalg.det(B))
print("det(A^2 B^-1) =", val, "   9/5 =", 9 / 5)
assert abs(val - 9 / 5) < 1e-12

det A = 3.0000000000000004  det B = 4.999999999999999
det(A^2 B^-1) = 1.8    9/5 = 1.8


### Problem L0.7 — Trace from a spectrum

**Statement.** A $3 \times 3$ matrix has eigenvalues $1, 2, -1$. Find $\operatorname{tr}(A)$ and
$\det(A)$.

**Intuition.** The trace is the sum of the roots of $p_A$ and the determinant is their product.

**Solution.**

*Step 1.* By Theorem 4.6, $\operatorname{tr}(A) = \sum_i \lambda_i = 1 + 2 - 1 = 2$.

*Step 2.* By the same theorem, $\det(A) = \prod_i \lambda_i = 1 \cdot 2 \cdot (-1) = -2$.

$$
\boxed{\operatorname{tr}(A) = 2, \qquad \det(A) = -2}
$$

**Key takeaway.** Two of the $n$ coefficients of $p_A$ are readable without ever computing
$p_A$.

In [8]:
lam = np.array([1.0, 2.0, -1.0])
Ad = np.diag(lam)
Q, _ = np.linalg.qr(rng.standard_normal((3, 3)))
Asim = Q @ Ad @ Q.T
print("trace:", np.trace(Asim), " sum lambda:", lam.sum())
print("det  :", np.linalg.det(Asim), " prod lambda:", np.prod(lam))
assert abs(np.trace(Asim) - 2.0) < 1e-12
assert abs(np.linalg.det(Asim) + 2.0) < 1e-12

trace: 1.9999999999999998  sum lambda: 2.0
det  : -2.0  prod lambda: -2.0


### Problem L0.8 — Scaling a $4 \times 4$ matrix

**Statement.** Let $A$ be $4 \times 4$ with $\det A = 2$. Find $\det(3A)$.

**Intuition.** Scaling by $3$ stretches all four axes, and each contributes a factor $3$ to the
volume.

**Solution.**

*Step 1.* Multilinearity lets the factor $3$ be pulled out of each of the four columns.

*Step 2.* By Theorem 4.3, $\det(3A) = 3^{4}\det(A) = 81 \cdot 2$.

$$
\boxed{\det(3A) = 162}
$$

**Key takeaway.** $\det(cA) = c^{n}\det(A)$: the exponent is the dimension, not $1$ and not $2$.

In [9]:
A = np.diag([2.0, 1.0, 1.0, 1.0])
print("det A =", np.linalg.det(A), "   det(3A) =", np.linalg.det(3 * A), "   3^4 * 2 =", 3 ** 4 * 2)
assert abs(np.linalg.det(3 * A) - 162.0) < 1e-10

det A = 2.0    det(3A) = 162.00000000000009    3^4 * 2 = 162


### Problem L0.9 — Characteristic polynomial of a triangular $2 \times 2$

**Statement.** Find $p_A(t)$ for $A = \begin{pmatrix} a & b \\ 0 & c \end{pmatrix}$.

**Intuition.** $tI - A$ stays triangular, and a triangular determinant is the diagonal product.

**Solution.**

*Step 1.* $tI - A = \begin{pmatrix} t-a & -b \\ 0 & t-c \end{pmatrix}$.

*Step 2.* By Problem L0.2 the determinant is the product of the diagonal.

$$
\boxed{p_A(t) = (t-a)(t-c)}
$$

**Key takeaway.** The eigenvalues of a triangular matrix sit on its diagonal, and $b$ never
appears — it affects the eigenvectors, not the spectrum.

In [10]:
for a, b, c in [(2.0, 5.0, 3.0), (1.0, 0.0, 1.0)]:
    A = np.array([[a, b], [0.0, c]])
    print(f"a={a}, b={b}, c={c}:  numpy poly {np.poly(A)}   expected {[1.0, -(a + c), a * c]}")
    assert np.allclose(np.poly(A), [1.0, -(a + c), a * c])

a=2.0, b=5.0, c=3.0:  numpy poly [ 1. -5.  6.]   expected [1.0, -5.0, 6.0]
a=1.0, b=0.0, c=1.0:  numpy poly [ 1. -2.  1.]   expected [1.0, -2.0, 1.0]


### Problem L0.10 — Shape of a Kronecker product

**Statement.** For $A \in \mathbb{R}^{3 \times 4}$ and $B \in \mathbb{R}^{2 \times 5}$, give the
shape of $A \otimes B$.

**Intuition.** Each entry of $A$ is replaced by a whole copy of $B$.

**Solution.**

*Step 1.* Definition 3.7 places a $2 \times 5$ block at each of the $3 \times 4$ positions.

*Step 2.* Rows: $3 \cdot 2 = 6$. Columns: $4 \cdot 5 = 20$.

$$
\boxed{A \otimes B \in \mathbb{R}^{6 \times 20}}
$$

**Key takeaway.** Dimensions multiply, which is why a Kronecker-structured object of moderate
factor size can be astronomically large if formed explicitly.

In [11]:
A = rng.standard_normal((3, 4))
B = rng.standard_normal((2, 5))
print("shape of A kron B:", np.kron(A, B).shape, "  = (3*2, 4*5)")
print("shape of B kron A:", np.kron(B, A).shape, "  same shape here, but")
print("A kron B equals B kron A ?", np.allclose(np.kron(A, B), np.kron(B, A)))
assert np.kron(A, B).shape == (6, 20)
assert not np.allclose(np.kron(A, B), np.kron(B, A))

shape of A kron B: (6, 20)   = (3*2, 4*5)
shape of B kron A: (6, 20)   same shape here, but
A kron B equals B kron A ? False


## L1 — Foundations

### Problem L1.1 — Trace of a commutator

**Statement.** Prove $\operatorname{tr}(AB - BA) = 0$ for square $A, B$ of the same size.

**Intuition.** The trace cannot tell $AB$ from $BA$.

**Solution.**

*Step 1.* Linearity of the trace (Theorem 4.5) gives
$\operatorname{tr}(AB - BA) = \operatorname{tr}(AB) - \operatorname{tr}(BA)$.

*Step 2.* The commutation rule of Theorem 4.5 gives
$\operatorname{tr}(AB) = \operatorname{tr}(BA)$.

$$
\boxed{\operatorname{tr}([A,B]) = 0}
$$

**Key takeaway.** No commutator equals a non-zero multiple of $I$ over a field of characteristic
zero, since $\operatorname{tr}(cI) = cn \neq 0$. Problem L3.13 makes that precise.

In [12]:
A = rng.standard_normal((5, 5))
B = rng.standard_normal((5, 5))
comm = A @ B - B @ A
print(f"tr(AB - BA) = {np.trace(comm):.3e} = {abs(np.trace(comm)) / EPS:.1f} * eps")
assert abs(np.trace(comm)) < 1e-12

tr(AB - BA) = 8.882e-16 = 4.0 * eps


### Problem L1.2 — Cayley-Hamilton for a $2 \times 2$ matrix

**Statement.** Write the Cayley-Hamilton identity for
$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$ and verify it.

**Intuition.** For $n = 2$ the characteristic polynomial has only trace and determinant in it.

**Solution.**

*Step 1.* $\operatorname{tr}(A) = 5$ and $\det(A) = 4 - 6 = -2$.

*Step 2.* Theorem 4.6 with $n = 2$ gives
$p_A(t) = t^{2} - \operatorname{tr}(A)t + \det(A) = t^{2} - 5t - 2$.

*Step 3.* Theorem 4.7 gives $p_A(A) = 0$. Checking directly,
$A^{2} = \begin{pmatrix} 7 & 10 \\ 15 & 22 \end{pmatrix}$, and

$$
A^{2} - 5A - 2I = \begin{pmatrix} 7 - 5 - 2 & 10 - 10 \\ 15 - 15 & 22 - 20 - 2 \end{pmatrix} = 0 .
$$

$$
\boxed{A^{2} - 5A - 2I = 0}
$$

**Key takeaway.** For every $2 \times 2$ matrix,
$A^{2} = \operatorname{tr}(A) A - \det(A) I$, so all powers of a $2 \times 2$ live in the plane
spanned by $I$ and $A$.

In [13]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
print("A^2 =\n", A @ A)
resid = A @ A - np.trace(A) * A + np.linalg.det(A) * np.eye(2)
print("A^2 - tr(A) A + det(A) I =\n", np.round(resid, 12))
print(f"Frobenius norm = {np.linalg.norm(resid):.3e}")
assert np.linalg.norm(resid) < 1e-12

A^2 =
 [[ 7. 10.]
 [15. 22.]]
A^2 - tr(A) A + det(A) I =
 [[-0.  0.]
 [ 0. -0.]]
Frobenius norm = 6.280e-16


### Problem L1.3 — Trace of an outer product

**Statement.** Prove $\operatorname{tr}(u v^{\top}) = v^{\top} u$ for $u, v \in \mathbb{R}^{n}$.

**Intuition.** The outer product is $n \times n$ and the inner product is $1 \times 1$, but the
trace does not care about the size.

**Solution.**

*Step 1.* Apply the commutation rule of Theorem 4.5 with $A = u \in \mathbb{R}^{n \times 1}$ and
$B = v^{\top} \in \mathbb{R}^{1 \times n}$:
$\operatorname{tr}(u v^{\top}) = \operatorname{tr}(v^{\top} u)$.

*Step 2.* $v^{\top}u$ is a $1 \times 1$ matrix, whose trace is its single entry.

$$
\boxed{\operatorname{tr}(u v^{\top}) = v^{\top} u = u^{\top} v}
$$

**Key takeaway.** This is the smallest instance of the trace trick: it converts an $n \times n$
object into a scalar without forming the matrix.

In [14]:
u = rng.standard_normal(6)
v = rng.standard_normal(6)
print(f"tr(u v^T) = {np.trace(np.outer(u, v)):.12f}")
print(f"v^T u     = {v @ u:.12f}")
assert abs(np.trace(np.outer(u, v)) - v @ u) < 1e-12

tr(u v^T) = -0.321879308063
v^T u     = -0.321879308063


### Problem L1.4 — Determinant of an orthogonal matrix

**Statement.** Let $Q^{\top}Q = I$. Find the possible values of $\det Q$, and say which sign
corresponds to a rotation.

**Intuition.** An isometry cannot change volume, so the factor must have modulus $1$.

**Solution.**

*Step 1.* Take determinants: $\det(Q^{\top}Q) = \det(I) = 1$.

*Step 2.* Theorem 4.3 splits the left side and Theorem 4.2 identifies the two factors:
$\det(Q^{\top})\det(Q) = \det(Q)^{2}$.

*Step 3.* Hence $\det(Q)^{2} = 1$.

$$
\boxed{\det Q = \pm 1}
$$

The value $+1$ is the rotation case (orientation preserved) and $-1$ is a reflection.

**Key takeaway.** Theorem 4.2 is doing real work here: without $\det(Q^{\top}) = \det(Q)$ the
argument does not close.

In [15]:
Qr, _ = np.linalg.qr(rng.standard_normal((4, 4)))
F = np.eye(4)
F[0, 0] = -1.0
print("random orthogonal Q :", np.linalg.det(Qr))
print("reflection F        :", np.linalg.det(F))
print("||Q^T Q - I||_F     :", np.linalg.norm(Qr.T @ Qr - np.eye(4)))
assert abs(abs(np.linalg.det(Qr)) - 1.0) < 1e-12
assert abs(np.linalg.det(F) + 1.0) < 1e-12

random orthogonal Q : -1.0000000000000004
reflection F        : -1.0
||Q^T Q - I||_F     : 6.644078455613987e-16


### Problem L1.5 — When a symmetric $2 \times 2$ is singular

**Statement.** For which $k$ is $A = \begin{pmatrix} 1 & k \\ k & 4 \end{pmatrix}$ singular? For
which $k$ is it positive definite?

**Intuition.** Singularity is the vanishing of one scalar.

**Solution.**

*Step 1.* $\det A = 4 - k^{2}$, so $A$ is singular exactly when $k^{2} = 4$.

*Step 2.* By Theorem 4.3 that is the invertibility criterion, so $k = \pm 2$.

*Step 3.* For definiteness, both leading principal minors must be positive: $1 \gt 0$ always,
and $4 - k^{2} \gt 0$ means $\lvert k \rvert \lt 2$.

$$
\boxed{\text{singular for } k = \pm 2; \quad A \succ 0 \text{ for } \lvert k \rvert \lt 2}
$$

**Key takeaway.** One determinant answers both questions, which is Sylvester's criterion in its
smallest case.

In [16]:
for k in (-3.0, -2.0, 0.0, 1.5, 2.0, 3.0):
    A = np.array([[1.0, k], [k, 4.0]])
    ev = np.linalg.eigvalsh(A)
    print(f"k = {k:+.1f}:  det = {np.linalg.det(A):+.2f}   eigenvalues = {ev}"
          f"   {'singular' if abs(np.linalg.det(A)) < 1e-12 else ('PD' if ev.min() > 0 else 'not PD')}")
assert abs(np.linalg.det(np.array([[1.0, 2.0], [2.0, 4.0]]))) < 1e-12

k = -3.0:  det = -5.00   eigenvalues = [-0.8541  5.8541]   not PD
k = -2.0:  det = +0.00   eigenvalues = [0. 5.]   singular
k = +0.0:  det = +4.00   eigenvalues = [1. 4.]   PD
k = +1.5:  det = +1.75   eigenvalues = [0.3787 4.6213]   PD
k = +2.0:  det = +0.00   eigenvalues = [0. 5.]   singular
k = +3.0:  det = -5.00   eigenvalues = [-0.8541  5.8541]   not PD


### Problem L1.6 — Frobenius norm through the trace

**Statement.** Show that $\lVert A \rVert_F^{2} = \sum_{i,j} A_{ij}^{2}$ equals
$\operatorname{tr}(A^{\top}A)$.

**Intuition.** The $j$-th diagonal entry of $A^{\top}A$ is the squared length of the $j$-th
column.

**Solution.**

*Step 1.* $(A^{\top}A)_{jj} = \sum_{i} (A^{\top})_{ji} A_{ij} = \sum_{i} A_{ij}^{2}$.

*Step 2.* Summing over $j$ gives $\sum_{i,j} A_{ij}^{2}$.

$$
\boxed{\lVert A \rVert_F^{2} = \operatorname{tr}(A^{\top}A)}
$$

**Key takeaway.** Definition 3.10 makes $\operatorname{tr}(A^{\top}B)$ an inner product; this is
the norm it induces, and it is the Euclidean norm of $\operatorname{vec}(A)$.

In [17]:
A = rng.standard_normal((4, 6))
print("sum of squares :", np.sum(A ** 2))
print("tr(A^T A)      :", np.trace(A.T @ A))
print("||vec(A)||^2   :", np.linalg.norm(A.reshape(-1, order="F")) ** 2)
assert abs(np.trace(A.T @ A) - np.sum(A ** 2)) < 1e-10

sum of squares : 25.479835255401184
tr(A^T A)      : 25.479835255401188
||vec(A)||^2   : 25.479835255401184


### Problem L1.7 — Gradient of a linear trace functional

**Statement.** Prove $\nabla_X \operatorname{tr}(AX) = A^{\top}$.

**Intuition.** $\operatorname{tr}(AX)$ is the Frobenius inner product of $A^{\top}$ with $X$, so
it is linear with that coefficient.

**Solution.**

*Step 1.* $\operatorname{tr}(AX) = \sum_{i} \sum_{k} A_{ik} X_{ki}$.

*Step 2.* Differentiating in the single variable $X_{ki}$ leaves $A_{ik}$.

*Step 3.* Assembling the array whose $(k,i)$ entry is $A_{ik}$ gives $A^{\top}$.

$$
\boxed{\nabla_X \operatorname{tr}(AX) = A^{\top}}
$$

**Key takeaway.** Equivalently $\operatorname{tr}(AX) = \langle A^{\top}, X \rangle_F$, which is
the matrix version of $\nabla_x (a^{\top}x) = a$ and the backward pass of a linear layer.

In [18]:
A = rng.standard_normal((3, 4))
X = rng.standard_normal((4, 3))
h = 1e-6
G = np.zeros_like(X)
for i in range(4):
    for j in range(3):
        E = np.zeros_like(X)
        E[i, j] = h
        G[i, j] = (np.trace(A @ (X + E)) - np.trace(A @ (X - E))) / (2 * h)
print("finite-difference gradient:\n", np.round(G, 6))
print("A^T:\n", np.round(A.T, 6))
print("max difference:", np.abs(G - A.T).max())
assert np.abs(G - A.T).max() < 1e-7

finite-difference gradient:
 [[-0.2976 -0.0498  0.9175]
 [-0.53    0.0866  1.0669]
 [-0.2362 -1.4871  0.0477]
 [ 1.8165  1.6473  0.9167]]
A^T:
 [[-0.2976 -0.0498  0.9175]
 [-0.53    0.0866  1.0669]
 [-0.2362 -1.4871  0.0477]
 [ 1.8165  1.6473  0.9167]]
max difference: 1.0643617931460625e-10


### Problem L1.8 — Sherman-Morrison, base case

**Statement.** Let $u, v \in \mathbb{R}^{n}$ with $1 + v^{\top}u \neq 0$. Verify

$$
(I + u v^{\top})^{-1} = I - \frac{u v^{\top}}{1 + v^{\top} u},
$$

and explain what happens when $1 + v^{\top}u = 0$.

**Intuition.** The inverse of a rank-one update is another rank-one update, because everything
happens in the line spanned by $u$.

**Solution.**

*Step 1.* Multiply out, writing $\alpha = 1 + v^{\top}u$:

$$
(I + uv^{\top})\left( I - \frac{uv^{\top}}{\alpha} \right)
= I + uv^{\top} - \frac{uv^{\top}}{\alpha} - \frac{u (v^{\top}u) v^{\top}}{\alpha} .
$$

*Step 2.* Since $v^{\top}u$ is a scalar, the last two terms combine:

$$
\frac{uv^{\top}(1 + v^{\top}u)}{\alpha} = u v^{\top} .
$$

*Step 3.* Hence the product is $I + uv^{\top} - uv^{\top} = I$.

*Step 4.* The hypothesis is not cosmetic. By Theorem 4.12,
$\det(I + uv^{\top}) = 1 + v^{\top}u$, so $\alpha = 0$ makes $I + uv^{\top}$ singular; indeed
$(I + uv^{\top})u = u(1 + v^{\top}u) = 0$, so $u$ lies in the null space.

$$
\boxed{(I + u v^{\top})^{-1} = I - \frac{u v^{\top}}{1 + v^{\top} u}, \quad \text{valid iff } 1 + v^{\top}u \neq 0}
$$

**Key takeaway.** A rank-one update costs $\mathcal{O}(n^{2})$ instead of $\mathcal{O}(n^{3})$,
and the determinant identity of Theorem 4.12 tells you in advance when the update is legal.

In [19]:
n = 5
u = rng.standard_normal(n)
v = rng.standard_normal(n)
alpha = 1 + v @ u
Minv = np.eye(n) - np.outer(u, v) / alpha
print(f"1 + v^T u = {alpha:.6f}   det(I + uv^T) = {np.linalg.det(np.eye(n) + np.outer(u, v)):.6f}")
print("residual ||(I+uv^T) M - I||_F =",
      np.linalg.norm((np.eye(n) + np.outer(u, v)) @ Minv - np.eye(n)))
assert np.linalg.norm((np.eye(n) + np.outer(u, v)) @ Minv - np.eye(n)) < 1e-10

vbad = -u / (u @ u)
print("\nsingular case: 1 + v^T u =", 1 + vbad @ u)
Sing = np.eye(n) + np.outer(u, vbad)
print("  det =", np.linalg.det(Sing), "   ||(I + u v^T) u|| =", np.linalg.norm(Sing @ u))
assert abs(np.linalg.det(Sing)) < 1e-12

1 + v^T u = 2.238634   det(I + uv^T) = 2.238634
residual ||(I+uv^T) M - I||_F = 4.1619473140396716e-16

singular case: 1 + v^T u = 0.0
  det = -7.907255461557456e-17    ||(I + u v^T) u|| = 1.3387743760663253e-16


### Problem L1.9 — Nilpotent matrices have zero trace

**Statement.** Prove that $A^{k} = 0$ for some $k \ge 1$ forces $\operatorname{tr}(A) = 0$, and
also $\det(A) = 0$ when $n \ge 1$.

**Intuition.** A nilpotent matrix can only have $0$ as a root of its characteristic polynomial.

**Solution.**

*Step 1.* If $\lambda$ is a root of $p_A$ then $A - \lambda I$ is singular, so there is
$v \neq 0$ with $Av = \lambda v$, hence $A^{k}v = \lambda^{k}v$.

*Step 2.* $A^{k} = 0$ gives $\lambda^{k}v = 0$ and therefore $\lambda = 0$.

*Step 3.* Every root of $p_A$ is $0$, so by Theorem 4.6
$\operatorname{tr}(A) = \sum_i \lambda_i = 0$ and $\det(A) = \prod_i \lambda_i = 0$; indeed
$p_A(t) = t^{n}$.

$$
\boxed{A \text{ nilpotent} \implies p_A(t) = t^{n}, \ \operatorname{tr}(A) = 0, \ \det(A) = 0}
$$

**Key takeaway.** Cayley-Hamilton then gives $A^{n} = 0$: nilpotency never needs an exponent
larger than $n$.

In [20]:
N = np.triu(rng.standard_normal((5, 5)), 1)
print("strictly upper triangular N, so N^5 = 0")
print("  trace           :", np.trace(N))
print("  det             :", np.linalg.det(N))
print("  char polynomial :", np.round(np.poly(N), 12))
print("  ||N^5||_F       :", np.linalg.norm(np.linalg.matrix_power(N, 5)))
assert abs(np.trace(N)) < 1e-12
assert np.linalg.norm(np.linalg.matrix_power(N, 5)) < 1e-12

strictly upper triangular N, so N^5 = 0
  trace           : 0.0
  det             : 0.0
  char polynomial : [1. 0. 0. 0. 0. 0.]
  ||N^5||_F       : 0.0


### Problem L1.10 — Determinant of a block triangular matrix

**Statement.** Prove
$\det \begin{pmatrix} A & B \\ 0 & C \end{pmatrix} = \det(A)\det(C)$ for square $A$ and $C$,
**without** assuming $A$ invertible.

**Intuition.** Freeze the bottom block and use the uniqueness half of Theorem 4.1 twice.

**Solution.**

*Step 1.* Let $A$ be $p \times p$ and $C$ be $q \times q$, and define

$$
F(A) = \det \begin{pmatrix} A & B \\ 0 & C \end{pmatrix}
$$

as a function of the first $p$ columns of the big matrix, with $B$ and $C$ held fixed.

*Step 2.* $F$ is multilinear in those columns, because $\det$ is, and it vanishes when two of
them coincide, because then the big matrix has two equal columns. Theorem 4.1 therefore gives
$F(A) = F(I_p)\det(A)$.

*Step 3.* Repeat on the last $q$ columns of
$\begin{pmatrix} I_p & B \\ 0 & C \end{pmatrix}$, which gives
$F(I_p) = \det \begin{pmatrix} I_p & B \\ 0 & I_q \end{pmatrix} \det(C)$.

*Step 4.* The remaining matrix is unitriangular, so the Leibniz formula leaves only the identity
permutation and its determinant is $1$.

$$
\boxed{\det \begin{pmatrix} A & B \\ 0 & C \end{pmatrix} = \det(A)\det(C)}
$$

**Key takeaway.** The usual factorization argument needs $A^{-1}$ and then patches the singular
case "by continuity", which only works over $\mathbb{R}$ or $\mathbb{C}$. This proof is valid
over every commutative ring and covers the singular case directly.

In [21]:
p, q = 3, 2
for label, Ablk in [("A invertible", rng.standard_normal((p, p))),
                    ("A singular", np.outer(rng.standard_normal(p), rng.standard_normal(p)))]:
    Bblk = rng.standard_normal((p, q))
    Cblk = rng.standard_normal((q, q))
    Mblk = np.block([[Ablk, Bblk], [np.zeros((q, p)), Cblk]])
    lhs = np.linalg.det(Mblk)
    rhs = np.linalg.det(Ablk) * np.linalg.det(Cblk)
    print(f"{label:<14s}  det M = {lhs:+.10f}   det A det C = {rhs:+.10f}   diff {abs(lhs - rhs):.2e}")
    assert abs(lhs - rhs) < 1e-9

A invertible    det M = +0.7035236042   det A det C = +0.7035236042   diff 1.11e-16
A singular      det M = -0.0000000000   det A det C = -0.0000000000   diff 7.55e-46


### Problem L1.11 — Inverse from an annihilating polynomial

**Statement.** Suppose $A^{3} - 4A^{2} + 3A - 2I = 0$. Express $A^{-1}$ as a polynomial in $A$.

**Intuition.** Isolate the $I$ term and factor one $A$ out of everything else.

**Solution.**

*Step 1.* Rearrange: $A^{3} - 4A^{2} + 3A = 2I$.

*Step 2.* Factor $A$ out on the left: $A(A^{2} - 4A + 3I) = 2I$.

*Step 3.* The relation shows $A$ has a right inverse, hence is invertible, and multiplying by
$A^{-1}$ gives $A^{2} - 4A + 3I = 2 A^{-1}$.

$$
\boxed{A^{-1} = \tfrac{1}{2}\bigl( A^{2} - 4A + 3I \bigr)}
$$

**Key takeaway.** Any annihilating polynomial with non-zero constant term produces the inverse.
Cayley-Hamilton (Theorem 4.7) guarantees one exists whenever $\det A \neq 0$.

In [22]:
C = np.array([[0.0, 0.0, 2.0], [1.0, 0.0, -3.0], [0.0, 1.0, 4.0]])
print("companion matrix of t^3 - 4t^2 + 3t - 2, char poly:", np.round(np.poly(C), 10))
resid = np.linalg.matrix_power(C, 3) - 4 * C @ C + 3 * C - 2 * np.eye(3)
print("annihilating check ||.||_F =", np.linalg.norm(resid))
guess = 0.5 * (C @ C - 4 * C + 3 * np.eye(3))
print("polynomial inverse:\n", np.round(guess, 8))
print("numpy inverse     :\n", np.round(np.linalg.inv(C), 8))
assert np.linalg.norm(resid) < 1e-10
assert np.allclose(guess, np.linalg.inv(C))

companion matrix of t^3 - 4t^2 + 3t - 2, char poly: [ 1. -4.  3. -2.]
annihilating check ||.||_F = 0.0
polynomial inverse:
 [[ 1.5  1.   0. ]
 [-2.   0.   1. ]
 [ 0.5  0.   0. ]]
numpy inverse     :
 [[ 1.5  1.   0. ]
 [-2.   0.   1. ]
 [ 0.5  0.   0. ]]


### Problem L1.12 — Trace of a Kronecker product

**Statement.** Prove $\operatorname{tr}(A \otimes B) = \operatorname{tr}(A)\operatorname{tr}(B)$
for square $A$ and $B$.

**Intuition.** The diagonal of $A \otimes B$ collects the diagonals of the diagonal blocks.

**Solution.**

*Step 1.* The diagonal blocks of $A \otimes B$ are $A_{ii}B$ for $i = 1, \dots, n$.

*Step 2.* The trace of the block $A_{ii}B$ is $A_{ii}\operatorname{tr}(B)$.

*Step 3.* Summing over $i$ factors the double sum.

$$
\boxed{\operatorname{tr}(A \otimes B) = \operatorname{tr}(A) \operatorname{tr}(B)}
$$

**Key takeaway.** The trace is multiplicative over $\otimes$ while it is additive over the direct
sum; the Kronecker product turns sums of dimensions into products.

In [23]:
A = rng.standard_normal((3, 3))
B = rng.standard_normal((4, 4))
print(f"tr(A kron B) = {np.trace(np.kron(A, B)):.12f}")
print(f"tr(A) tr(B)  = {np.trace(A) * np.trace(B):.12f}")
assert abs(np.trace(np.kron(A, B)) - np.trace(A) * np.trace(B)) < 1e-10

tr(A kron B) = -1.102416538757
tr(A) tr(B)  = -1.102416538757


### Problem L1.13 — Minimal polynomial strictly smaller than the characteristic one

**Statement.** Exhibit a matrix whose minimal polynomial is a proper divisor of its
characteristic polynomial, and one where the two agree.

**Intuition.** Repeated eigenvalues with a full set of independent eigenvectors let the minimal
polynomial keep only simple roots.

**Solution.**

*Step 1.* Take $A = I_2$. Then $p_A(t) = \det(tI - I) = (t-1)^{2}$.

*Step 2.* But $A - I = 0$, so $m(t) = t - 1$ annihilates and no constant polynomial does.
Hence $m_A(t) = t - 1$, a proper divisor of $p_A$.

*Step 3.* By contrast take $N = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$. Again
$p_N(t) = (t-1)^{2}$, but $N - I = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix} \neq 0$, so
$m_N(t) = (t-1)^{2} = p_N(t)$.

$$
\boxed{m_{I_2}(t) = t - 1 \ \big| \ p_{I_2}(t) = (t-1)^2, \qquad m_N = p_N = (t-1)^2}
$$

**Key takeaway.** The two matrices have the same characteristic polynomial and different minimal
polynomials, so $p_A$ alone does not determine similarity class.

In [24]:
I2 = np.eye(2)
N = np.array([[1.0, 1.0], [0.0, 1.0]])
for M, name in [(I2, "I2"), (N, "shear")]:
    print(f"{name}:  char poly {np.poly(M)}   ||M - I||_F = {np.linalg.norm(M - np.eye(2)):.1f}"
          f"   ||(M - I)^2||_F = {np.linalg.norm((M - np.eye(2)) @ (M - np.eye(2))):.1f}")
assert np.linalg.norm(I2 - np.eye(2)) == 0.0
assert np.linalg.norm(N - np.eye(2)) > 0.5
assert np.linalg.norm((N - np.eye(2)) @ (N - np.eye(2))) < 1e-12

I2:  char poly [ 1. -2.  1.]   ||M - I||_F = 0.0   ||(M - I)^2||_F = 0.0
shear:  char poly [ 1. -2.  1.]   ||M - I||_F = 1.0   ||(M - I)^2||_F = 0.0


### Problem L1.14 — Exponential of a nilpotent matrix

**Statement.** If $A^{3} = 0$, compute $e^{A}$ exactly, and check $\det(e^{A}) = 1$.

**Intuition.** The exponential series stops as soon as the powers vanish.

**Solution.**

*Step 1.* $e^{A} = \sum_{k \ge 0} A^{k}/k!$ by definition.

*Step 2.* $A^{k} = A^{k-3}A^{3} = 0$ for every $k \ge 3$, so only three terms survive.

*Step 3.* By Problem L1.9, $\operatorname{tr}(A) = 0$, so Theorem 4.9 gives
$\det(e^{A}) = e^{0} = 1$.

$$
\boxed{e^{A} = I + A + \tfrac{1}{2}A^{2}, \qquad \det(e^{A}) = 1}
$$

**Key takeaway.** Nilpotent generators produce volume-preserving, polynomial flows: the shear
$e^{tN}$ is the standard example.

In [25]:
from scipy.linalg import expm

A = np.array([[0.0, 2.0, 3.0], [0.0, 0.0, 4.0], [0.0, 0.0, 0.0]])
print("A^3 =\n", np.linalg.matrix_power(A, 3).astype(int))
closed = np.eye(3) + A + 0.5 * A @ A
print("I + A + A^2/2 =\n", closed)
print("scipy expm(A) =\n", expm(A))
print(f"max difference = {np.abs(closed - expm(A)).max():.3e}")
print(f"det(exp A) = {np.linalg.det(expm(A)):.12f}   exp(tr A) = {np.exp(np.trace(A)):.12f}")
assert np.allclose(closed, expm(A))
assert abs(np.linalg.det(expm(A)) - 1.0) < 1e-12

A^3 =
 [[0 0 0]
 [0 0 0]
 [0 0 0]]
I + A + A^2/2 =
 [[1. 2. 7.]
 [0. 1. 4.]
 [0. 0. 1.]]
scipy expm(A) =
 [[1. 2. 7.]
 [0. 1. 4.]
 [0. 0. 1.]]
max difference = 0.000e+00
det(exp A) = 1.000000000000   exp(tr A) = 1.000000000000


### Problem L1.15 — Determinant of a rank-one update

**Statement.** Let $A = u v^{\top}$ with $u, v \in \mathbb{R}^{n}$. Compute $\det(I + A)$, and
then $\det(X + uv^{\top})$ for invertible $X$.

**Intuition.** A rank-one update changes the volume only along one direction.

**Solution.**

*Step 1.* Apply Theorem 4.12 with the $n \times 1$ matrix $u$ and the $1 \times n$ matrix
$v^{\top}$:

$$
\det(I_n + u v^{\top}) = \det(I_1 + v^{\top}u) = 1 + v^{\top}u .
$$

*Step 2.* For invertible $X$, factor $X + uv^{\top} = X\bigl( I + X^{-1}u v^{\top} \bigr)$ and
apply Step 1 with $u$ replaced by $X^{-1}u$.

$$
\boxed{\det(I + uv^{\top}) = 1 + v^{\top}u, \qquad \det(X + uv^{\top}) = \det(X)\bigl(1 + v^{\top}X^{-1}u\bigr)}
$$

**Key takeaway.** This is the matrix determinant lemma. It is what makes a rank-one Gaussian
covariance update, or a BFGS step, cost $\mathcal{O}(n^{2})$ rather than $\mathcal{O}(n^{3})$.

In [26]:
n = 5
u = rng.standard_normal(n)
v = rng.standard_normal(n)
X = rng.standard_normal((n, n)) + 3 * np.eye(n)
print(f"det(I + uv^T)      = {np.linalg.det(np.eye(n) + np.outer(u, v)):.10f}")
print(f"1 + v^T u          = {1 + v @ u:.10f}")
lhs = np.linalg.det(X + np.outer(u, v))
rhs = np.linalg.det(X) * (1 + v @ np.linalg.solve(X, u))
print(f"det(X + uv^T)      = {lhs:.10f}")
print(f"det X (1+v^T X^-1 u) = {rhs:.10f}")
assert abs(np.linalg.det(np.eye(n) + np.outer(u, v)) - (1 + v @ u)) < 1e-10
assert abs(lhs - rhs) / abs(rhs) < 1e-10

det(I + uv^T)      = -0.1365446767
1 + v^T u          = -0.1365446767
det(X + uv^T)      = 98.0790733396
det X (1+v^T X^-1 u) = 98.0790733396


### Problem L1.16 — An eigenvector of a Kronecker product

**Statement.** If $Ax = \lambda x$ and $By = \mu y$ with $x, y \neq 0$, show that $x \otimes y$
is an eigenvector of $A \otimes B$, and say why this alone does not determine the full spectrum.

**Intuition.** The mixed-product property lets the two factors act independently.

**Solution.**

*Step 1.* Read $x$ and $y$ as one-column matrices and apply Theorem 4.11:

$$
(A \otimes B)(x \otimes y) = (Ax) \otimes (By) = (\lambda x) \otimes (\mu y) = \lambda\mu \, (x \otimes y).
$$

*Step 2.* $x \otimes y \neq 0$ because both factors are non-zero, so it is a genuine eigenvector.

*Step 3.* What this does **not** show is that these are all the eigenvalues with the right
multiplicities. If $A$ or $B$ is defective there are fewer than $nm$ vectors of the form
$x \otimes y$, and the counting argument needs the triangularization of Proof 5.11.

$$
\boxed{(A \otimes B)(x \otimes y) = \lambda\mu \, (x \otimes y)}
$$

**Key takeaway.** Producing an eigenvalue is easy; proving you have found them all is the
theorem. This is exactly the gap Proof 5.11 closes.

In [27]:
A = np.array([[2.0, 1.0], [0.0, 3.0]])
B = np.array([[0.0, -1.0], [1.0, 0.0]])
wa, Va = np.linalg.eig(A)
wb, Vb = np.linalg.eig(B)
K = np.kron(A, B)
for i in range(2):
    for j in range(2):
        vec = np.kron(Va[:, i], Vb[:, j])
        res = np.linalg.norm(K @ vec - wa[i] * wb[j] * vec)
        print(f"lambda = {wa[i]:+.3f}, mu = {wb[j]:+.3f}  ->  residual {res:.3e}")
        assert res < 1e-12
print("spectrum of A kron B:", np.sort_complex(np.linalg.eigvals(K)))
print("all products        :", np.sort_complex(np.array([a * b for a in wa for b in wb])))

lambda = +2.000, mu = +0.000+1.000j  ->  residual 0.000e+00
lambda = +2.000, mu = +0.000-1.000j  ->  residual 0.000e+00
lambda = +3.000, mu = +0.000+1.000j  ->  residual 0.000e+00
lambda = +3.000, mu = +0.000-1.000j  ->  residual 0.000e+00
spectrum of A kron B: [0.-3.j 0.-2.j 0.+2.j 0.+3.j]
all products        : [0.-3.j 0.-2.j 0.+2.j 0.+3.j]


### Problem L1.17 — The $3 \times 3$ Vandermonde determinant

**Statement.** Compute

$$
\det V, \qquad V = \begin{pmatrix} 1 & 1 & 1 \\ x_1 & x_2 & x_3 \\ x_1^{2} & x_2^{2} & x_3^{2} \end{pmatrix},
$$

and evaluate it at $(x_1, x_2, x_3) = (1, 2, 4)$.

**Intuition.** The determinant must vanish whenever two nodes coincide, so it is divisible by
each difference $x_j - x_i$.

**Solution.**

*Step 1.* Subtract $x_1$ times row $2$ from row $3$, and $x_1$ times row $1$ from row $2$;
neither operation changes the determinant. The first column becomes $(1, 0, 0)^{\top}$ and

$$
\det V = \det \begin{pmatrix} x_2 - x_1 & x_3 - x_1 \\ x_2(x_2 - x_1) & x_3(x_3 - x_1) \end{pmatrix}.
$$

*Step 2.* Pull $x_2 - x_1$ out of the first column and $x_3 - x_1$ out of the second:

$$
\det V = (x_2 - x_1)(x_3 - x_1) \det \begin{pmatrix} 1 & 1 \\ x_2 & x_3 \end{pmatrix} = (x_2 - x_1)(x_3 - x_1)(x_3 - x_2) .
$$

*Step 3.* At $(1, 2, 4)$: $(2-1)(4-1)(4-2) = 1 \cdot 3 \cdot 2 = 6$.

$$
\boxed{\det V = \prod_{1 \le i \lt j \le 3} (x_j - x_i), \qquad \det V \big|_{(1,2,4)} = 6}
$$

**Key takeaway.** $\det V \neq 0$ exactly when the nodes are distinct, which is why polynomial
interpolation at distinct points always has a unique solution.

In [28]:
x = np.array([1.0, 2.0, 4.0])
V = np.vander(x, increasing=True).T
print("V =\n", V)
formula = prod(x[j] - x[i] for i in range(3) for j in range(i + 1, 3))
print(f"det V = {np.linalg.det(V):.10f}   product of differences = {formula:.10f}")
xr = rng.standard_normal(3)
Vr = np.vander(xr, increasing=True).T
fr = prod(xr[j] - xr[i] for i in range(3) for j in range(i + 1, 3))
print(f"random nodes: det = {np.linalg.det(Vr):.10f}   formula = {fr:.10f}")
assert abs(np.linalg.det(V) - 6.0) < 1e-10
assert abs(np.linalg.det(Vr) - fr) < 1e-10

V =


 [[ 1.  1.  1.]
 [ 1.  2.  4.]
 [ 1.  4. 16.]]
det V = 6.0000000000   product of differences = 6.0000000000
random nodes: det = 0.2355501962   formula = 0.2355501962


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Explained variance in PCA

**Statement.** A covariance matrix $\Sigma \in \mathbb{R}^{n \times n}$ has eigenvalues
$\lambda_1 \ge \cdots \ge \lambda_n \ge 0$. Write the fraction of total variance captured by the
first $k$ principal components, and evaluate it for
$\Sigma = \operatorname{diag}(6, 3, 2, 1)$ with $k = 2$.

**Intuition.** Variance adds across orthogonal directions, and the trace is that total.

**Solution.**

*Step 1.* Total variance is $\operatorname{tr}(\Sigma) = \sum_{i=1}^{n}\lambda_i$ by
Theorem 4.6.

*Step 2.* The first $k$ components capture $\sum_{i=1}^{k}\lambda_i$.

*Step 3.* For $\Sigma = \operatorname{diag}(6,3,2,1)$: total $= 12$, first two $= 9$, ratio
$= 3/4$.

$$
\boxed{\frac{\sum_{i=1}^{k}\lambda_i}{\sum_{i=1}^{n}\lambda_i}, \qquad \text{here } \frac{9}{12} = 0.75}
$$

**Key takeaway.** The trace is the normalizing constant of every scree plot, and it is available
without diagonalizing anything.

In [29]:
Sigma = np.diag([6.0, 3.0, 2.0, 1.0])
lam = np.linalg.eigvalsh(Sigma)[::-1]
print("eigenvalues :", lam, "  total variance = trace =", np.trace(Sigma))
for k in range(1, 5):
    print(f"  k = {k}:  explained fraction = {lam[:k].sum() / lam.sum():.4f}")
assert abs(lam[:2].sum() / np.trace(Sigma) - 0.75) < 1e-12

eigenvalues :

 [6. 3. 2. 1.]   total variance = trace = 12.0
  k = 1:  explained fraction = 0.5000
  k = 2:  explained fraction = 0.7500
  k = 3:  explained fraction = 0.9167
  k = 4:  explained fraction = 1.0000


### Problem L2.2 — The trace trick for a quadratic form

**Statement.** Let $x$ have mean $0$ and covariance $\Sigma = \mathbb{E}[xx^{\top}]$. Prove
$\mathbb{E}[x^{\top}Ax] = \operatorname{tr}(A\Sigma)$.

**Intuition.** A scalar equals its own trace, and then the trace can be rotated.

**Solution.**

*Step 1.* $x^{\top}Ax$ is a $1 \times 1$ matrix, so
$x^{\top}Ax = \operatorname{tr}(x^{\top}Ax)$.

*Step 2.* Commutation (Theorem 4.5) moves $x^{\top}$ to the back:
$\operatorname{tr}(x^{\top}Ax) = \operatorname{tr}(A x x^{\top})$.

*Step 3.* The trace is linear, so it commutes with expectation:
$\mathbb{E}[\operatorname{tr}(Axx^{\top})] = \operatorname{tr}(A\,\mathbb{E}[xx^{\top}])$.

$$
\boxed{\mathbb{E}[x^{\top} A x] = \operatorname{tr}(A\Sigma)}
$$

**Key takeaway.** The identity turns an expectation over vectors into one matrix product; it is
the backbone of risk formulas, LQR cost-to-go, and the Hutchinson estimator of Problem L2.11.

In [30]:
Sig = np.array([[2.0, 0.5], [0.5, 1.0]])
A = np.array([[1.0, -2.0], [3.0, 0.5]])
L = np.linalg.cholesky(Sig)
Xs = (L @ rng.standard_normal((2, 400000))).T
emp = np.mean(np.einsum("ij,jk,ik->i", Xs, A, Xs))
print(f"empirical mean of x^T A x : {emp:.6f}")
print(f"tr(A Sigma)               : {np.trace(A @ Sig):.6f}")
assert abs(emp - np.trace(A @ Sig)) < 0.05

empirical mean of x^T A x : 3.000718
tr(A Sigma)               : 3.000000


### Problem L2.3 — Determinant of a covariance and Gaussian entropy

**Statement.** The differential entropy of $\mathcal{N}(\mu, \Sigma)$ in $\mathbb{R}^{n}$ is
$h = \tfrac12 \ln \det (2\pi e \Sigma)$. Rewrite it in terms of $\ln\det\Sigma$ and describe what
happens as $\det\Sigma \to 0$.

**Intuition.** The determinant is the volume of the uncertainty ellipsoid, and a flat ellipsoid
carries no volume.

**Solution.**

*Step 1.* By Theorem 4.3, $\det(2\pi e \Sigma) = (2\pi e)^{n}\det\Sigma$, so

$$
h = \tfrac{n}{2}\ln(2\pi e) + \tfrac12 \ln\det\Sigma .
$$

*Step 2.* $\det\Sigma \to 0$ means some eigenvalue tends to $0$: the distribution concentrates
on a hyperplane.

*Step 3.* Then $\ln\det\Sigma \to -\infty$ and $h \to -\infty$.

$$
\boxed{h = \tfrac{n}{2}\ln(2\pi e) + \tfrac12 \ln\det\Sigma \ \longrightarrow \ -\infty \text{ as } \det\Sigma \to 0}
$$

**Key takeaway.** Differential entropy is unbounded below, and the determinant is exactly the
quantity that detects the collapse. This is why maximum-likelihood covariance estimates are
regularized as $\Sigma + \varepsilon I$.

In [31]:
n = 3
for eps in (1.0, 1e-2, 1e-6):
    Sig = np.diag([1.0, 1.0, eps])
    h = 0.5 * np.log(np.linalg.det(2 * np.pi * np.e * Sig))
    h2 = 0.5 * n * np.log(2 * np.pi * np.e) + 0.5 * np.log(np.linalg.det(Sig))
    print(f"smallest eigenvalue {eps:8.1e}:  h = {h:+10.6f}   rewritten {h2:+10.6f}")
    assert abs(h - h2) < 1e-9

smallest eigenvalue  1.0e+00:  h =  +4.256816   rewritten  +4.256816
smallest eigenvalue  1.0e-02:  h =  +1.954231   rewritten  +1.954231
smallest eigenvalue  1.0e-06:  h =  -2.650940   rewritten  -2.650940


### Problem L2.4 — Hadamard gating in an LSTM

**Statement.** An LSTM cell updates its state as
$C_t = f_t \circ C_{t-1} + i_t \circ \tilde{C}_t$ with gates $f_t, i_t \in (0,1)^{n}$. Why is the
Hadamard product used rather than a matrix product?

**Intuition.** A gate is a per-coordinate volume knob, not a mixing operator.

**Solution.**

*Step 1.* $(f_t \circ C_{t-1})_j = (f_t)_j (C_{t-1})_j$: coordinate $j$ of the new state depends
only on coordinate $j$ of the old one.

*Step 2.* A matrix product $W C_{t-1}$ would mix all coordinates, so a single gate value could
no longer be read as "how much of memory slot $j$ to keep".

*Step 3.* The gradient through the recurrence is therefore
$\partial C_t/\partial C_{t-1} = \operatorname{diag}(f_t)$, whose determinant is $\prod_j (f_t)_j$
and whose spectrum is $\{(f_t)_j\}$. Keeping some gates near $1$ keeps that Jacobian from
shrinking, which is precisely the vanishing-gradient fix.

$$
\boxed{\text{element-wise gating keeps the recurrent Jacobian diagonal: } \partial C_t/\partial C_{t-1} = \operatorname{diag}(f_t)}
$$

**Key takeaway.** The design choice is a statement about a Jacobian. Diagonal Jacobians have
determinants you can read off, which is also why coupling-layer normalizing flows are built the
same way.

In [32]:
f = np.array([0.99, 0.95, 0.10])
C0 = np.array([1.0, 1.0, 1.0])
state = C0.copy()
print("diagonal recurrent Jacobian: det =", np.prod(f))
for t in (1, 10, 50):
    print(f"  after {t:2d} steps, state = {C0 * f ** t}   det of Jacobian product = {np.prod(f) ** t:.3e}")
assert abs(np.prod(f) - np.linalg.det(np.diag(f))) < 1e-12

diagonal recurrent Jacobian: det = 0.09405000000000001
  after  1 steps, state = [0.99 0.95 0.1 ]   det of Jacobian product = 9.405e-02
  after 10 steps, state = [0.9044 0.5987 0.    ]   det of Jacobian product = 5.415e-11
  after 50 steps, state = [0.605  0.0769 0.    ]   det of Jacobian product = 4.655e-52


### Problem L2.5 — Gradient of the log-determinant

**Statement.** Prove $\nabla_X \ln\det X = X^{-\top}$ for invertible $X$, and specialize to
symmetric $X$.

**Intuition.** It is the matrix version of $\frac{d}{dx}\ln x = 1/x$.

**Solution.**

*Step 1.* By Proof 5.9, $\partial \det X/\partial X_{ij} = (\operatorname{adj}X)_{ji}$.

*Step 2.* Chain rule on the logarithm:

$$
\frac{\partial \ln\det X}{\partial X_{ij}} = \frac{(\operatorname{adj}X)_{ji}}{\det X} = (X^{-1})_{ji},
$$

using $\operatorname{adj}X = \det(X) X^{-1}$ from Theorem 4.4.

*Step 3.* Assembling the array whose $(i,j)$ entry is $(X^{-1})_{ji}$ gives $X^{-\top}$; for
symmetric $X$ this is $X^{-1}$.

$$
\boxed{\nabla_X \ln \det X = X^{-\top}, \qquad = X^{-1} \text{ when } X = X^{\top}}
$$

**Key takeaway.** Maximum-likelihood fitting of a Gaussian differentiates exactly this term, and
the answer being the inverse is why the MLE covariance solves $\Sigma^{-1} = \Sigma^{-1}S\Sigma^{-1}$.

In [33]:
n = 4
Xm = rng.standard_normal((n, n)) + 3 * np.eye(n)
h = 1e-6
G = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        E = np.zeros((n, n))
        E[i, j] = h
        G[i, j] = (np.log(abs(np.linalg.det(Xm + E))) - np.log(abs(np.linalg.det(Xm - E)))) / (2 * h)
print("finite differences:\n", np.round(G, 6))
print("X^{-T}:\n", np.round(np.linalg.inv(Xm).T, 6))
print("max difference:", np.abs(G - np.linalg.inv(Xm).T).max())
assert np.abs(G - np.linalg.inv(Xm).T).max() < 1e-6

finite differences:
 [[ 0.2963 -0.0419 -0.0415  0.0415]
 [ 0.0328  0.1715 -0.024  -0.0996]
 [-0.1049 -0.0843  0.3042  0.0124]
 [-0.0337  0.1116  0.0341  0.326 ]]
X^{-T}:
 [[ 0.2963 -0.0419 -0.0415  0.0415]
 [ 0.0328  0.1715 -0.024  -0.0996]
 [-0.1049 -0.0843  0.3042  0.0124]
 [-0.0337  0.1116  0.0341  0.326 ]]
max difference: 3.780000409903117e-10


### Problem L2.6 — The softmax Jacobian as matrix products

**Statement.** The softmax output $p$ has Jacobian $\operatorname{diag}(p) - p p^{\top}$. Write
it using the products of Definition 3.7, and show that it is singular.

**Intuition.** A diagonal term plus a rank-one correction.

**Solution.**

*Step 1.* $\operatorname{diag}(p)$ isolates the entries of $p$ onto the diagonal, which is the
Hadamard product $I \circ (p \mathbf{1}^{\top})$.

*Step 2.* The second term is the outer product $p p^{\top}$.

*Step 3.* Singularity: $J \mathbf{1} = p - p (\mathbf{1}^{\top}p) = p - p = 0$ since the entries
of $p$ sum to $1$. Hence $\det J = 0$ and $\mathbf{1}$ spans a null direction.

$$
\boxed{J = I \circ (p \mathbf{1}^{\top}) - p p^{\top}, \qquad J \mathbf{1} = 0, \ \det J = 0}
$$

**Key takeaway.** The softmax is invariant under adding a constant to all logits, and the zero
determinant of its Jacobian is that invariance made visible.

In [34]:
z = np.array([1.0, -0.5, 2.0, 0.3])
p = np.exp(z - z.max())
p /= p.sum()
J = np.diag(p) - np.outer(p, p)
J_alt = np.eye(4) * np.outer(p, np.ones(4)) - np.outer(p, p)
print("p =", p, " sum =", p.sum())
print("||J - (I o p 1^T - p p^T)||_F =", np.linalg.norm(J - J_alt))
print("J 1 =", np.round(J @ np.ones(4), 12), "   det J =", np.linalg.det(J))
assert np.allclose(J, J_alt)
assert abs(np.linalg.det(J)) < 1e-12

p =

 [0.2253 0.0503 0.6125 0.1119]  sum = 1.0
||J - (I o p 1^T - p p^T)||_F = 0.0
J 1 = [ 0. -0.  0.  0.]    det J = -1.0775036073326234e-20


### Problem L2.7 — Linearizing the Sylvester equation

**Statement.** Convert $AX + XB = C$ into a standard linear system, and state exactly when it
has a unique solution.

**Intuition.** Vectorize $X$ and let the vec identity of Theorem 4.11 do the bookkeeping.

**Solution.**

*Step 1.* $\operatorname{vec}(AX) = \operatorname{vec}(A X I) = (I \otimes A)\operatorname{vec}(X)$.

*Step 2.* $\operatorname{vec}(XB) = \operatorname{vec}(I X B) = (B^{\top} \otimes I)\operatorname{vec}(X)$.

*Step 3.* Adding gives the linear system

$$
\bigl( I \otimes A + B^{\top} \otimes I \bigr) \operatorname{vec}(X) = \operatorname{vec}(C) .
$$

*Step 4.* The coefficient matrix is a **Kronecker sum**, and its eigenvalues are the pairwise
sums $\lambda_i(A) + \mu_j(B)$; the system is non-singular exactly when none of them is zero.

$$
\boxed{\bigl( I \otimes A + B^{\top} \otimes I \bigr)\operatorname{vec}(X) = \operatorname{vec}(C), \quad \text{unique iff } \lambda_i(A) + \mu_j(B) \neq 0 \ \forall i,j}
$$

**Key takeaway.** The Lyapunov equation $AX + XA^{\top} = -Q$ is the special case
$B = A^{\top}$; it is solvable whenever $A$ has no two eigenvalues summing to zero, which holds
for every stable $A$.

In [35]:
from scipy.linalg import solve_sylvester

nA, nB = 3, 2
A = rng.standard_normal((nA, nA))
B = rng.standard_normal((nB, nB))
C = rng.standard_normal((nA, nB))
M = np.kron(np.eye(nB), A) + np.kron(B.T, np.eye(nA))
Xvec = np.linalg.solve(M, C.reshape(-1, order="F"))
Xkron = Xvec.reshape(nA, nB, order="F")
Xlib = solve_sylvester(A, B, C)
print("||A X + X B - C||_F  (Kronecker route) =", np.linalg.norm(A @ Xkron + Xkron @ B - C))
print("||X_kron - X_scipy||_F                 =", np.linalg.norm(Xkron - Xlib))
sums = np.array([a + b for a in np.linalg.eigvals(A) for b in np.linalg.eigvals(B)])
print("eigenvalues of the Kronecker sum:", np.sort_complex(np.linalg.eigvals(M)))
print("pairwise sums lambda + mu       :", np.sort_complex(sums))
assert np.linalg.norm(A @ Xkron + Xkron @ B - C) < 1e-9
assert np.allclose(np.sort_complex(np.linalg.eigvals(M)), np.sort_complex(sums))

||A X + X B - C||_F  (Kronecker route) = 1.0577751464707461e-14
||X_kron - X_scipy||_F                 = 1.125711400913677e-13


eigenvalues of the Kronecker sum: [-3.6692+0.j -2.6823+0.j -1.8299+0.j -1.7525+0.j -0.8431+0.j  0.0868+0.j]
pairwise sums lambda + mu       : [-3.6692+0.j -2.6823+0.j -1.8299+0.j -1.7525+0.j -0.8431+0.j  0.0868+0.j]


### Problem L2.8 — Khatri-Rao in the CP tensor decomposition

**Statement.** In the CP model the mode-$1$ unfolding is approximated by
$X_{(1)} \approx A (C \ast B)^{\top}$ with $C \in \mathbb{R}^{K \times R}$ and
$B \in \mathbb{R}^{J \times R}$. Give the shape of $C \ast B$, and prove the identity

$$
(C \ast B)^{\top} (C \ast B) = \bigl( C^{\top}C \bigr) \circ \bigl( B^{\top}B \bigr) .
$$

**Intuition.** The Khatri-Rao product turns a product of two inner products into one inner
product in a bigger space, exactly as in Proof 5.13.

**Solution.**

*Step 1.* The $r$-th column of $C \ast B$ is $c_r \otimes b_r \in \mathbb{R}^{KJ}$, and there
are $R$ columns, so $C \ast B \in \mathbb{R}^{KJ \times R}$.

*Step 2.* The $(r,s)$ entry of $(C \ast B)^{\top}(C \ast B)$ is
$(c_r \otimes b_r)^{\top}(c_s \otimes b_s)$.

*Step 3.* By the mixed-product property of Theorem 4.11 applied to row vectors,

$$
(c_r \otimes b_r)^{\top}(c_s \otimes b_s) = (c_r^{\top}c_s)(b_r^{\top}b_s) = (C^{\top}C)_{rs} \, (B^{\top}B)_{rs},
$$

which is the $(r,s)$ entry of $(C^{\top}C) \circ (B^{\top}B)$.

$$
\boxed{C \ast B \in \mathbb{R}^{KJ \times R}, \qquad (C \ast B)^{\top}(C \ast B) = (C^{\top}C) \circ (B^{\top}B)}
$$

**Key takeaway.** The identity replaces a $KJ \times R$ Gram computation by two small ones and a
Hadamard product, which is what makes ALS for CP decomposition affordable. It also shows the Gram
matrix is a Hadamard product of two positive semidefinite matrices, hence positive semidefinite
by Theorem 4.13 — as it must be.

In [36]:
K, J, R = 5, 4, 3
Cm = rng.standard_normal((K, R))
Bm = rng.standard_normal((J, R))
KR = np.column_stack([np.kron(Cm[:, r], Bm[:, r]) for r in range(R)])
print("shape of C * B :", KR.shape, "  expected", (K * J, R))
lhs = KR.T @ KR
rhs = (Cm.T @ Cm) * (Bm.T @ Bm)
print("||(C*B)^T (C*B) - (C^T C) o (B^T B)||_F =", np.linalg.norm(lhs - rhs))
print("eigenvalues of the Gram matrix           :", np.linalg.eigvalsh(lhs))
assert KR.shape == (K * J, R)
assert np.linalg.norm(lhs - rhs) < 1e-10
assert np.linalg.eigvalsh(lhs).min() > -1e-10

shape of C * B : (20, 3)   expected (20, 3)
||(C*B)^T (C*B) - (C^T C) o (B^T B)||_F = 1.4342141464242164e-14
eigenvalues of the Gram matrix           : [ 2.8934  6.9168 57.9132]


### Problem L2.9 — K-FAC without forming the Fisher matrix

**Statement.** K-FAC approximates the Fisher matrix of a linear layer as
$F \approx \Sigma_x \otimes \Sigma_g$ with $\Sigma_x = \mathbb{E}[xx^{\top}]$ and
$\Sigma_g = \mathbb{E}[gg^{\top}]$. Compute $F^{-1}\operatorname{vec}(G)$ efficiently and give the
cost.

**Intuition.** The inverse of a Kronecker product is the Kronecker product of the inverses.

**Solution.**

*Step 1.* The mixed-product property of Theorem 4.11 gives
$(\Sigma_x \otimes \Sigma_g)(\Sigma_x^{-1} \otimes \Sigma_g^{-1}) = I$, so
$F^{-1} = \Sigma_x^{-1} \otimes \Sigma_g^{-1}$.

*Step 2.* The vec identity with $A = \Sigma_g^{-1}$ and $B^{\top} = \Sigma_x^{-1}$ gives

$$
\bigl( \Sigma_x^{-1} \otimes \Sigma_g^{-1} \bigr)\operatorname{vec}(G) = \operatorname{vec}\bigl( \Sigma_g^{-1} G \, \Sigma_x^{-\top} \bigr),
$$

and $\Sigma_x$ is symmetric, so $\Sigma_x^{-\top} = \Sigma_x^{-1}$.

*Step 3.* Cost: forming and inverting $F$ is $\mathcal{O}(q^{3}p^{3})$ for
$\Sigma_x \in \mathbb{R}^{q \times q}$ and $\Sigma_g \in \mathbb{R}^{p \times p}$; the right-hand
side costs $\mathcal{O}(q^{3} + p^{3})$.

$$
\boxed{F^{-1}\operatorname{vec}(G) = \operatorname{vec}\bigl( \Sigma_g^{-1} G \Sigma_x^{-1} \bigr)}
$$

**Key takeaway.** For a layer with $p = q = 1000$ this replaces a $10^{6} \times 10^{6}$ inverse
by two $1000 \times 1000$ ones — the difference between impossible and routine.

In [37]:
q, p = 5, 4
Sx = rng.standard_normal((q, q)); Sx = Sx @ Sx.T + q * np.eye(q)
Sg = rng.standard_normal((p, p)); Sg = Sg @ Sg.T + p * np.eye(p)
Gw = rng.standard_normal((p, q))
F = np.kron(Sx, Sg)
direct = np.linalg.solve(F, Gw.reshape(-1, order="F"))
smart = (np.linalg.solve(Sg, Gw) @ np.linalg.inv(Sx)).reshape(-1, order="F")
print(f"F is {F.shape[0]}x{F.shape[1]};  factors are {q}x{q} and {p}x{p}")
print(f"relative difference = {np.linalg.norm(direct - smart) / np.linalg.norm(smart):.3e}")
print(f"det F = {np.linalg.det(F):.6e}   (det Sx)^p (det Sg)^q = "
      f"{np.linalg.det(Sx) ** p * np.linalg.det(Sg) ** q:.6e}")
assert np.linalg.norm(direct - smart) / np.linalg.norm(smart) < 1e-9

F is 20x20;  factors are 5x5 and 4x4


relative difference = 4.560e-16
det F = 8.445131e+36   (det Sx)^p (det Sg)^q = 8.445131e+36


### Problem L2.10 — The resolvent as a Neumann series

**Statement.** The resolvent of $A$ is $R(z) = (zI - A)^{-1}$. Expand it as a power series valid
for $\lvert z \rvert \gt \lVert A \rVert$, and say what the sharp radius is.

**Intuition.** It is the geometric series $1/(z - a) = \sum_k a^{k}/z^{k+1}$ with $a$ replaced by
a matrix.

**Solution.**

*Step 1.* Factor out $z$: $zI - A = z\bigl( I - \tfrac{1}{z}A \bigr)$, so
$R(z) = \tfrac{1}{z}\bigl( I - \tfrac{1}{z}A \bigr)^{-1}$.

*Step 2.* If $\lVert A/z \rVert \lt 1$ the Neumann series
$\bigl( I - M \bigr)^{-1} = \sum_{k \ge 0} M^{k}$ converges absolutely.

*Step 3.* Substituting $M = A/z$,

$$
R(z) = \frac{1}{z}\sum_{k \ge 0} \frac{A^{k}}{z^{k}} = \sum_{k \ge 0} \frac{A^{k}}{z^{k+1}} .
$$

*Step 4.* The sharp condition is $\lvert z \rvert \gt \rho(A)$, the spectral radius, because
$R(z)$ is analytic exactly off the spectrum, and $\rho(A) \le \lVert A \rVert$ always.

$$
\boxed{R(z) = \sum_{k \ge 0} \frac{A^{k}}{z^{k+1}}, \qquad \text{convergent for } \lvert z \rvert \gt \rho(A)}
$$

**Key takeaway.** $\det(zI - A) = p_A(z)$ vanishes exactly at the poles of $R$, so the resolvent
is the analytic object whose singularities are the spectrum — the starting point of the
holomorphic functional calculus and of Green's functions in physics. The cell below shows the
gap between the two radii: at $z = 0.58$, which lies between $\rho(A)$ and $\lVert A \rVert$,
the series still converges, only slowly.

In [38]:
A = np.array([[0.5, 0.2], [0.1, 0.3]])
rho = max(abs(np.linalg.eigvals(A)))
print(f"||A||_op = {np.linalg.norm(A, 2):.6f}   rho(A) = {rho:.6f}")

def neumann(A, z, K):
    """Partial sum of A^k / z^{k+1}, accumulated term by term to avoid overflow."""
    term = np.eye(A.shape[0]) / z
    total = term.copy()
    for _ in range(1, K):
        term = term @ A / z
        total = total + term
    return total, np.linalg.norm(term)

for z, K in [(2.0, 80), (0.58, 200), (0.58, 800), (0.40, 40)]:
    R_exact = np.linalg.inv(z * np.eye(2) - A)
    partial, last = neumann(A, z, K)
    print(f"z = {z:.2f}, K = {K:4d}:  |z| > rho ? {str(z > rho):<5s}"
          f"  ||R - partial sum||_F = {np.linalg.norm(R_exact - partial):.3e}"
          f"   last term = {last:.3e}")

conv, _ = neumann(A, 2.0, 80)
assert np.linalg.norm(np.linalg.inv(2.0 * np.eye(2) - A) - conv) < 1e-12
slow_200, _ = neumann(A, 0.58, 200)
slow_800, _ = neumann(A, 0.58, 800)
assert (np.linalg.norm(np.linalg.inv(0.58 * np.eye(2) - A) - slow_800)
        < np.linalg.norm(np.linalg.inv(0.58 * np.eye(2) - A) - slow_200))
_, blow = neumann(A, 0.40, 40)
assert blow > 1e5

||A||_op = 0.583390   rho(A) = 0.573205
z = 2.00, K =   80:  |z| > rho ? True   ||R - partial sum||_F = 1.121e-16   last term = 6.942e-44
z = 0.58, K =  200:  |z| > rho ? True   ||R - partial sum||_F = 1.451e+01   last term = 1.720e-01
z = 0.58, K =  800:  |z| > rho ? True   ||R - partial sum||_F = 1.233e-02   last term = 1.461e-04
z = 0.40, K =   40:  |z| > rho ? False  ||R - partial sum||_F = 1.069e+07   last term = 3.229e+06


### Problem L2.11 — The Hutchinson trace estimator

**Statement.** Let $z$ have independent entries with $\mathbb{E}[z_i] = 0$ and
$\mathbb{E}[z_i^{2}] = 1$. Show $\mathbb{E}[z^{\top}Mz] = \operatorname{tr}(M)$, and measure how
the error of the $m$-sample average decays.

**Intuition.** The identity of Problem L2.2 with $\Sigma = I$.

**Solution.**

*Step 1.* $\mathbb{E}[zz^{\top}] = I$ because the entries are uncorrelated with unit variance.

*Step 2.* By Problem L2.2, $\mathbb{E}[z^{\top}Mz] = \operatorname{tr}(M \cdot I) = \operatorname{tr}(M)$.

*Step 3.* Averaging $m$ independent probes gives an unbiased estimator whose standard deviation
falls as $m^{-1/2}$, the ordinary Monte-Carlo rate.

$$
\boxed{\widehat{\operatorname{tr}}(M) = \frac{1}{m}\sum_{s=1}^{m} z_s^{\top} M z_s, \qquad \text{error} = \Theta(m^{-1/2})}
$$

**Key takeaway.** Only matrix-vector products with $M$ are needed, never $M$ itself. That is what
makes continuous normalizing flows and log-determinant estimation possible at scale, since
Theorem 4.9 turns the $\log\det$ into a trace.

In [39]:
n = 40
Mh = rng.standard_normal((n, n))
Mh = Mh @ Mh.T / n
tr_true = np.trace(Mh)
ms = np.array([4, 16, 64, 256, 1024])
rms = []
for m in ms:
    errs = [np.mean(np.einsum("ij,ij->j", Z, Mh @ Z)) - tr_true
            for Z in (rng.choice([-1.0, 1.0], size=(n, m)) for _ in range(200))]
    rms.append(np.sqrt(np.mean(np.square(errs))))
rms = np.array(rms)
slope = np.polyfit(np.log(ms), np.log(rms), 1)[0]
print(f"true trace = {tr_true:.6f}")
for m, r in zip(ms, rms):
    print(f"  m = {m:5d}   rms error = {r:.6f}")
print(f"fitted exponent = {slope:.4f}   predicted -0.5")
assert abs(slope + 0.5) < 0.08

true trace = 40.259852
  m =     4   rms error = 4.315821
  m =    16   rms error = 2.038931
  m =    64   rms error = 1.087313
  m =   256   rms error = 0.530363
  m =  1024   rms error = 0.280175
fitted exponent = -0.4917   predicted -0.5


### Problem L2.12 — Liouville's theorem for a linear Hamiltonian flow (physics)

**Statement.** A harmonic oscillator of mass $m$ and stiffness $k$ has
$H(q,p) = \tfrac{p^{2}}{2m} + \tfrac{k}{2}q^{2}$ and phase-space equations
$\dot{q} = p/m$, $\dot{p} = -kq$. Show that the flow preserves phase-space volume, and compute
the volume factor when linear damping $-\gamma p$ is added.

**Intuition.** The trace of the generator is the rate of volume change.

**Solution.**

*Step 1.* The system is $\dot{x} = Mx$ with
$M = \begin{pmatrix} 0 & 1/m \\ -k & 0 \end{pmatrix}$ and $\operatorname{tr}(M) = 0$.

*Step 2.* The fundamental matrix satisfies $\dot{\Phi} = M\Phi$, $\Phi(0) = I$. Theorem 4.9 with
cyclic invariance gives

$$
\frac{d}{dt}\det\Phi = \det\Phi \cdot \operatorname{tr}\bigl(\Phi^{-1}M\Phi\bigr) = \det\Phi \cdot \operatorname{tr}(M) .
$$

*Step 3.* Hence $\det\Phi(t) = e^{t\operatorname{tr}M} = e^{0} = 1$: areas in the $(q,p)$ plane
are preserved.

*Step 4.* With damping, $M_\gamma = \begin{pmatrix} 0 & 1/m \\ -k & -\gamma \end{pmatrix}$ has
$\operatorname{tr} = -\gamma$, so $\det\Phi(t) = e^{-\gamma t}$.

$$
\boxed{\det\Phi(t) = e^{t\operatorname{tr}(M)}: \ 1 \text{ for the conservative flow}, \ e^{-\gamma t} \text{ with damping}}
$$

**Key takeaway.** Liouville's theorem is one line of determinant calculus. The general Hamiltonian
case is the same statement, because the Jacobian of a Hamiltonian vector field is
$J\nabla^{2}H$ and $\operatorname{tr}(J\nabla^{2}H) = 0$ for the symplectic $J$.

In [40]:
from scipy.linalg import expm

m_mass, k_stiff, gamma = 2.0, 8.0, 0.4
M0 = np.array([[0.0, 1.0 / m_mass], [-k_stiff, 0.0]])
Mg = np.array([[0.0, 1.0 / m_mass], [-k_stiff, -gamma]])
print(f"tr(M0) = {np.trace(M0):+.3f}   tr(M_gamma) = {np.trace(Mg):+.3f}")
for t in (0.5, 2.0, 5.0):
    print(f"  t = {t:4.1f}:  det Phi0 = {np.linalg.det(expm(M0 * t)):.12f}"
          f"   det Phi_gamma = {np.linalg.det(expm(Mg * t)):.12f}"
          f"   exp(-gamma t) = {np.exp(-gamma * t):.12f}")
    assert abs(np.linalg.det(expm(M0 * t)) - 1.0) < 1e-10
    assert abs(np.linalg.det(expm(Mg * t)) - np.exp(-gamma * t)) < 1e-10

tr(M0) = +0.000   tr(M_gamma) = -0.400
  t =  0.5:  det Phi0 = 1.000000000000   det Phi_gamma = 0.818730753078   exp(-gamma t) = 0.818730753078
  t =  2.0:  det Phi0 = 1.000000000000   det Phi_gamma = 0.449328964117   exp(-gamma t) = 0.449328964117
  t =  5.0:  det Phi0 = 1.000000000000   det Phi_gamma = 0.135335283237   exp(-gamma t) = 0.135335283237


### Problem L2.13 — Wronskian and Abel's identity (physics)

**Statement.** For $y'' + p(t) y' + q(t) y = 0$ with solutions $y_1, y_2$, the Wronskian is
$W = y_1 y_2' - y_2 y_1'$. Prove $W(t) = W(0)\exp\bigl(-\int_0^{t} p\bigr)$, and evaluate it for
the damped oscillator $y'' + 2\zeta\omega y' + \omega^{2}y = 0$.

**Intuition.** The Wronskian is the determinant of the fundamental matrix, so Jacobi's formula
applies.

**Solution.**

*Step 1.* Write the equation as the first-order system $\dot{x} = M(t)x$ with
$x = (y, y')^{\top}$ and

$$
M(t) = \begin{pmatrix} 0 & 1 \\ -q(t) & -p(t) \end{pmatrix}, \qquad \operatorname{tr}M(t) = -p(t) .
$$

*Step 2.* The matrix $\Phi = \begin{pmatrix} y_1 & y_2 \\ y_1' & y_2' \end{pmatrix}$ satisfies
$\dot{\Phi} = M\Phi$, and $\det\Phi = W$.

*Step 3.* Theorem 4.9 gives $\dot{W} = W \operatorname{tr}(M) = -p(t) W$, a scalar linear
equation.

*Step 4.* Integrating, $W(t) = W(0)\exp\bigl(-\int_0^{t}p(s)\,ds\bigr)$. For the damped
oscillator $p = 2\zeta\omega$ is constant, so $W(t) = W(0)e^{-2\zeta\omega t}$.

$$
\boxed{W(t) = W(0)\,e^{-\int_0^{t} p}, \qquad W(t) = W(0) e^{-2\zeta\omega t} \text{ for constant damping}}
$$

**Key takeaway.** Either $W$ vanishes identically or it never vanishes, so linear independence of
two solutions can be tested at a single point. Abel's identity is Liouville's theorem for a
scalar second-order equation.

In [41]:
from scipy.integrate import solve_ivp

omega, zeta = 3.0, 0.15
p_coef, q_coef = 2 * zeta * omega, omega ** 2

def rhs(t, Y):
    y1, v1, y2, v2 = Y
    return [v1, -p_coef * v1 - q_coef * y1, v2, -p_coef * v2 - q_coef * y2]

sol = solve_ivp(rhs, (0.0, 4.0), [1.0, 0.0, 0.0, 1.0], rtol=1e-11, atol=1e-13,
                t_eval=np.linspace(0.0, 4.0, 5))
W = sol.y[0] * sol.y[3] - sol.y[2] * sol.y[1]
print(f"p = {p_coef:.3f}, so Abel predicts W(t) = W(0) exp(-{p_coef:.3f} t)")
for t, w in zip(sol.t, W):
    print(f"  t = {t:.1f}:  W = {w:.12f}   prediction = {np.exp(-p_coef * t):.12f}")
assert np.allclose(W, np.exp(-p_coef * sol.t), rtol=1e-7)

p = 0.900, so Abel predicts W(t) = W(0) exp(-0.900 t)
  t = 0.0:  W = 1.000000000000   prediction = 1.000000000000
  t = 1.0:  W = 0.406569659738   prediction = 0.406569659741
  t = 2.0:  W = 0.165298888220   prediction = 0.165298888222
  t = 3.0:  W = 0.067205512739   prediction = 0.067205512740
  t = 4.0:  W = 0.027323722447   prediction = 0.027323722447


### Problem L2.14 — Purity of a quantum density matrix (physics)

**Statement.** A density matrix $\rho$ is Hermitian, positive semidefinite, with
$\operatorname{tr}\rho = 1$. Show that $\operatorname{tr}(\rho^{2}) \le 1$ with equality exactly
for a pure state $\rho = \psi\psi^{\ast}$, and evaluate both cases for a qubit.

**Intuition.** Purity is the squared Frobenius norm of $\rho$, and the trace constraint bounds
it.

**Solution.**

*Step 1.* $\rho$ is Hermitian positive semidefinite, so its eigenvalues $\lambda_i$ are real and
non-negative, and $\sum_i \lambda_i = \operatorname{tr}\rho = 1$ by Theorem 4.6.

*Step 2.* By the same theorem applied to $\rho^{2}$,
$\operatorname{tr}(\rho^{2}) = \sum_i \lambda_i^{2}$.

*Step 3.* For non-negative numbers summing to $1$, $\sum_i \lambda_i^{2} \le \sum_i \lambda_i = 1$,
because $\lambda_i^{2} \le \lambda_i$ when $0 \le \lambda_i \le 1$. Equality forces every
$\lambda_i \in \{0, 1\}$, hence exactly one eigenvalue equal to $1$: a rank-one projector
$\rho = \psi\psi^{\ast}$.

*Step 4.* Qubit examples: the pure state
$\rho_{\text{pure}} = \tfrac12 \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$ has eigenvalues
$1, 0$ and purity $1$; the mixed state $\rho_{\text{mix}} = \operatorname{diag}(0.7, 0.3)$ has
purity $0.49 + 0.09 = 0.58$.

$$
\boxed{\operatorname{tr}(\rho^{2}) \le 1, \text{ with equality iff } \rho \text{ is a rank-one projector}}
$$

**Key takeaway.** Trace and determinant are the physical read-outs: $\operatorname{tr}\rho = 1$ is
conservation of probability, $\operatorname{tr}\rho^{2}$ measures purity, and $\det\rho = 0$ for
every pure state of dimension at least two.

In [42]:
rho_pure = 0.5 * np.array([[1.0, 1.0], [1.0, 1.0]])
rho_mix = np.diag([0.7, 0.3])
rho_max = 0.5 * np.eye(2)
for rho, name in [(rho_pure, "pure"), (rho_mix, "mixed"), (rho_max, "maximally mixed")]:
    ev = np.linalg.eigvalsh(rho)
    print(f"{name:<16s} tr = {np.trace(rho):.4f}   eigenvalues = {ev}"
          f"   purity = {np.trace(rho @ rho):.4f}   det = {np.linalg.det(rho):.4f}")
    assert abs(np.trace(rho) - 1.0) < 1e-12
    assert np.trace(rho @ rho) <= 1.0 + 1e-12
assert abs(np.trace(rho_pure @ rho_pure) - 1.0) < 1e-12
assert abs(np.trace(rho_mix @ rho_mix) - 0.58) < 1e-12

pure             tr = 1.0000   eigenvalues = [0. 1.]   purity = 1.0000   det = 0.0000
mixed            tr = 1.0000   eigenvalues = [0.3 0.7]   purity = 0.5800   det = 0.2100
maximally mixed  tr = 1.0000   eigenvalues = [0.5 0.5]   purity = 0.5000   det = 0.2500


## L3 — Challenge Proofs

### Problem L3.1 — Jacobi's formula and the second derivative of $\log\det$

**Statement.** Prove the differential form $d(\det A) = \operatorname{tr}(\operatorname{adj}(A)\, dA)$,
and use it to show that for $X \succ 0$ and symmetric $H$,

$$
\frac{d^{2}}{dt^{2}} \log\det(X + tH) = -\operatorname{tr}\bigl( M^{-1} H M^{-1} H \bigr), \qquad M = X + tH .
$$

**Intuition.** The determinant is a polynomial whose partial derivatives are the cofactors; one
more differentiation needs the derivative of a matrix inverse.

**Solution.**

*Step 1 — the partial derivatives.* Laplace expansion along row $i$ (Theorem 4.4) writes
$\det A = \sum_{k} A_{ik}C_{ik}$, and no cofactor $C_{ik}$ contains an entry of row $i$.
Differentiating in the single variable $A_{ij}$,

$$
\frac{\partial \det A}{\partial A_{ij}} = C_{ij} = (\operatorname{adj}A)_{ji} .
$$

*Step 2 — assemble.* Summing the chain rule over all entries,

$$
d(\det A) = \sum_{i,j} (\operatorname{adj}A)_{ji} \, dA_{ij} = \operatorname{tr}\bigl( \operatorname{adj}(A)\, dA \bigr).
$$

*Step 3 — first derivative of $\log\det$.* For invertible $M$, $\operatorname{adj}M = \det(M)M^{-1}$,
so $\frac{d}{dt}\log\det M = \operatorname{tr}(M^{-1}\dot{M})$. With $M = X + tH$ this is
$\operatorname{tr}(M^{-1}H)$.

*Step 4 — derivative of an inverse.* Differentiate $M M^{-1} = I$:
$\dot{M}M^{-1} + M \frac{d}{dt}M^{-1} = 0$, hence

$$
\frac{d}{dt} M^{-1} = -M^{-1}\dot{M}M^{-1} = -M^{-1} H M^{-1} .
$$

*Step 5 — second derivative.* Differentiating $\operatorname{tr}(M^{-1}H)$ and using linearity of
the trace,

$$
\frac{d^{2}}{dt^{2}} \log\det(X + tH) = \operatorname{tr}\bigl( -M^{-1}HM^{-1} H \bigr) = -\operatorname{tr}\bigl( M^{-1}HM^{-1}H \bigr).
$$

$$
\boxed{d(\det A) = \operatorname{tr}\bigl(\operatorname{adj}(A)\,dA\bigr), \qquad \frac{d^{2}}{dt^{2}}\log\det(X+tH) = -\operatorname{tr}\bigl(M^{-1}HM^{-1}H\bigr)}
$$

**Key takeaway.** The adjugate form is valid even at singular $A$, because both sides are
polynomials in the entries. Problem L3.12 turns the second derivative into a convexity statement.

In [43]:
n = 4
X = rng.standard_normal((n, n))
X = X @ X.T + n * np.eye(n)
H = rng.standard_normal((n, n))
H = (H + H.T) / 2

def g(t):
    return np.log(np.linalg.det(X + t * H))

h = 1e-4
num2 = (g(h) - 2 * g(0.0) + g(-h)) / h ** 2
Xi = np.linalg.inv(X)
ana2 = -np.trace(Xi @ H @ Xi @ H)
print(f"central second difference : {num2:.8f}")
print(f"-tr(X^-1 H X^-1 H)        : {ana2:.8f}")
print(f"absolute difference       : {abs(num2 - ana2):.3e}")
assert abs(num2 - ana2) < 1e-4

central second difference : -0.20621505
-tr(X^-1 H X^-1 H)        : -0.20621484
absolute difference       : 2.093e-07


### Problem L3.2 — Cayley-Hamilton fails over a non-commutative ring

**Statement.** Does Cayley-Hamilton hold when the entries of $A$ lie in a non-commutative ring?
Identify the step of Proof 5.7 that fails and give an explicit counterexample.

**Intuition.** The proof multiplies a matrix of polynomials by $tI - A$ and compares
coefficients, which silently assumes that $t$ commutes with every entry.

**Solution.**

*Step 1 — locate the failure.* Step 1 of Proof 5.7 uses the adjugate identity
$(tI - A)\operatorname{adj}(tI - A) = p_A(t) I$, which is a polynomial identity valid over
*commutative* rings only. Over a non-commutative ring the determinant itself has no
multiplicative, well-defined analogue, so already $p_A$ is ambiguous.

*Step 2 — the concrete test.* Over the quaternions $\mathbb{H}$, take the diagonal
$2 \times 2$ matrix

$$
A = \begin{pmatrix} i & 0 \\ 0 & j \end{pmatrix},
$$

and form the naive characteristic polynomial
$t^{2} - \operatorname{tr}(A)t + \det(A)$ with $\operatorname{tr}(A) = i + j$ and
$\det(A) = ij = k$, multiplying on the left.

*Step 3 — evaluate.* $A^{2} = \operatorname{diag}(i^{2}, j^{2}) = -I$. Next,

$$
(i+j)i = i^{2} + ji = -1 - k, \qquad (i+j)j = ij + j^{2} = k - 1,
$$

so $(i+j)A = \operatorname{diag}(-1-k, \, -1+k)$. Therefore

$$
A^{2} - (i+j)A + k I = \operatorname{diag}\bigl( -1 + 1 + k + k, \ -1 + 1 - k + k \bigr) = \operatorname{diag}(2k, 0) \neq 0 .
$$

*Step 4 — what survives.* There are genuine non-commutative substitutes, for instance the Study
determinant on $\mathbb{H}^{n \times n}$, obtained by passing to the $2n \times 2n$ complex
representation; the naive coefficientwise version is simply false.

$$
\boxed{\text{No: } A = \operatorname{diag}(i,j) \text{ gives } A^{2} - (i+j)A + ijI = \operatorname{diag}(2k, 0) \neq 0}
$$

**Key takeaway.** Commutativity is not a technical convenience in Proof 5.7; it is the hypothesis.
The counterexample is a $2 \times 2$ matrix, so the failure appears at the smallest interesting
size.

In [44]:
qi = np.array([[1j, 0], [0, -1j]])
qj = np.array([[0, 1 + 0j], [-1, 0]])
qk = qi @ qj
Z2 = np.zeros((2, 2), dtype=complex)
print("check that k = ij and that the three anticommute:")
print("  ji + ij =", np.round(qj @ qi + qi @ qj, 10).tolist())

Aq = np.block([[qi, Z2], [Z2, qj]])
Tq = np.block([[qi + qj, Z2], [Z2, qi + qj]])
Dq = np.block([[qk, Z2], [Z2, qk]])
resid = Aq @ Aq - Tq @ Aq + Dq
print("residual, top-left quaternion block:\n", np.round(resid[:2, :2], 10))
print("bottom-right block:\n", np.round(resid[2:, 2:], 10))
print("the quaternion 2k:\n", np.round(2 * qk, 10))
print(f"residual Frobenius norm = {np.linalg.norm(resid):.6f}")
assert np.allclose(resid[:2, :2], 2 * qk)
assert np.allclose(resid[2:, 2:], 0)

check that k = ij and that the three anticommute:
  ji + ij = [[0j, 0j], [0j, 0j]]
residual, top-left quaternion block:
 [[0.+0.j 0.+2.j]
 [0.+2.j 0.+0.j]]
bottom-right block:
 [[0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
the quaternion 2k:
 [[0.+0.j 0.+2.j]
 [0.+2.j 0.+0.j]]
residual Frobenius norm = 2.828427


### Problem L3.3 — Determinant of a Schur complement

**Statement.** For $M = \begin{pmatrix} A & B \\ C & D \end{pmatrix}$ with $D$ invertible, prove

$$
\det M = \det(D) \, \det\bigl( A - B D^{-1} C \bigr).
$$

**Intuition.** Block Gaussian elimination clears $B$ and $C$ using unit-determinant factors.

**Solution.**

*Step 1 — factor.* Verify the identity by multiplying out:

$$
\begin{pmatrix} A & B \\ C & D \end{pmatrix}
= \begin{pmatrix} I & B D^{-1} \\ 0 & I \end{pmatrix}
\begin{pmatrix} A - B D^{-1} C & 0 \\ 0 & D \end{pmatrix}
\begin{pmatrix} I & 0 \\ D^{-1} C & I \end{pmatrix}.
$$

The $(1,1)$ block of the product is $(A - BD^{-1}C) + BD^{-1}C = A$, the $(1,2)$ block is
$BD^{-1}D = B$, the $(2,1)$ block is $DD^{-1}C = C$, and the $(2,2)$ block is $D$.

*Step 2 — take determinants.* By Theorem 4.3 the determinant is multiplicative, and by the block
triangular rule the two outer factors are unitriangular with determinant $1$.

*Step 3 — the middle factor.* It is block diagonal, so its determinant is
$\det(A - BD^{-1}C)\det(D)$ by the same rule.

$$
\boxed{\det M = \det(D)\det\bigl(A - BD^{-1}C\bigr)}
$$

**Key takeaway.** The Schur complement $A - BD^{-1}C$ is the covariance of the conditional
Gaussian $x \mid y$, so this identity is the determinant bookkeeping behind
$p(x, y) = p(x \mid y)p(y)$.

In [45]:
pd, qd = 3, 2
Mfull = rng.standard_normal((pd + qd, pd + qd))
A = Mfull[:pd, :pd]; B = Mfull[:pd, pd:]
C = Mfull[pd:, :pd]; D = Mfull[pd:, pd:]
schur = A - B @ np.linalg.solve(D, C)
print(f"det M                       = {np.linalg.det(Mfull):+.10f}")
print(f"det D * det(A - B D^-1 C)   = {np.linalg.det(D) * np.linalg.det(schur):+.10f}")
assert abs(np.linalg.det(Mfull) - np.linalg.det(D) * np.linalg.det(schur)) < 1e-9

det M                       = -0.5909714768
det D * det(A - B D^-1 C)   = -0.5909714768


### Problem L3.4 — $\det A = \exp(\operatorname{tr}\log A)$

**Statement.** Let $A$ be diagonalizable with all eigenvalues positive. Prove
$\det A = \exp(\operatorname{tr}\log A)$, where $\log A = V (\log\Lambda) V^{-1}$ for any
diagonalization $A = V\Lambda V^{-1}$.

**Intuition.** The logarithm turns the product of eigenvalues into their sum.

**Solution.**

*Step 1 — the logarithm is well defined.* If $A = V\Lambda V^{-1}$ with
$\Lambda = \operatorname{diag}(\lambda_i)$ and $\lambda_i \gt 0$, set
$\log A = V \operatorname{diag}(\log\lambda_i) V^{-1}$. Then $e^{\log A} = V\Lambda V^{-1} = A$,
because the exponential of a diagonalizable matrix acts diagonally.

*Step 2 — trace of the logarithm.* Similarity invariance (Theorem 4.5) gives

$$
\operatorname{tr}(\log A) = \operatorname{tr}\bigl( \operatorname{diag}(\log \lambda_i) \bigr) = \sum_{i} \log \lambda_i .
$$

*Step 3 — apply the exponential identity.* By Theorem 4.9,
$\det(e^{X}) = e^{\operatorname{tr}X}$. Taking $X = \log A$,

$$
\det A = \det\bigl( e^{\log A} \bigr) = e^{\operatorname{tr}\log A} = \exp\left( \sum_i \log\lambda_i \right) = \prod_i \lambda_i,
$$

which agrees with Theorem 4.6.

$$
\boxed{\det A = \exp\bigl( \operatorname{tr}\log A \bigr)}
$$

**Key takeaway.** The identity converts a product over $n$ eigenvalues into a sum, which is what
makes $\log\det$ the numerically safe quantity — and what lets a stochastic trace estimator
(Problem L2.11) reach a determinant it could never compute directly.

In [46]:
from scipy.linalg import logm, expm

n = 4
Ap = rng.standard_normal((n, n))
Ap = Ap @ Ap.T + n * np.eye(n)
logA = logm(Ap).real
print("eigenvalues:", np.linalg.eigvalsh(Ap))
print(f"log det A          = {np.log(np.linalg.det(Ap)):.12f}")
print(f"tr(log A)          = {np.trace(logA):.12f}")
print(f"sum log lambda_i   = {np.log(np.linalg.eigvalsh(Ap)).sum():.12f}")
print(f"||exp(log A) - A|| = {np.linalg.norm(expm(logA) - Ap):.3e}")
assert abs(np.trace(logA) - np.log(np.linalg.det(Ap))) < 1e-9

eigenvalues: [ 4.0432  4.2649  9.5278 12.7898]
log det A          = 7.650315096881
tr(log A)          = 7.650315096881
sum log lambda_i   = 7.650315096881
||exp(log A) - A|| = 2.272e-13


### Problem L3.5 — Hadamard's inequality

**Statement.** Prove that for every $A \in \mathbb{R}^{n \times n}$ with columns $a_1, \dots, a_n$,

$$
\lvert \det A \rvert \le \prod_{j=1}^{n} \lVert a_j \rVert_2,
$$

with equality exactly when the columns are pairwise orthogonal or some column is zero.

**Intuition.** A box of given edge lengths has the largest volume when the edges are
perpendicular.

**Solution.**

*Step 1 — singular case.* If $A$ is singular, $\det A = 0$ and the inequality is immediate.

*Step 2 — QR factorization.* Otherwise $A$ has full column rank, so
[Module 04](../04_orthogonality_projections_and_qr/) provides $A = QR$ with $Q$ orthogonal and
$R$ upper triangular with positive diagonal.

*Step 3 — the determinant.* $\det A = \det(Q)\det(R) = \pm \prod_{j} r_{jj}$ by Theorem 4.3 and
Problem L0.2, so $\lvert \det A \rvert = \prod_j r_{jj}$.

*Step 4 — the column norms.* Since $a_j = Q r_j$ with $r_j$ the $j$-th column of $R$, and $Q$
preserves norms,

$$
\lVert a_j \rVert^{2} = \lVert r_j \rVert^{2} = \sum_{i \le j} r_{ij}^{2} \ \ge \ r_{jj}^{2} .
$$

*Step 5 — multiply.* Taking the product over $j$ gives
$\prod_j \lVert a_j \rVert \ge \prod_j r_{jj} = \lvert\det A\rvert$.

*Step 6 — equality.* Equality forces $r_{ij} = 0$ for all $i \lt j$, so $R$ is diagonal and
$A = QR$ has orthogonal columns.

$$
\boxed{\lvert \det A \rvert \le \prod_{j=1}^{n} \lVert a_j \rVert_2}
$$

**Key takeaway.** The inequality can be extremely loose: the $3 \times 3$ matrix with rows
$(1,1,1)$, $(1,2,3)$, $(1,3,6)$ has determinant $1$ against a bound near $44$. Tightness is
exactly orthogonality.

In [47]:
Hm = np.array([[1.0, 1.0, 1.0], [1.0, 2.0, 3.0], [1.0, 3.0, 6.0]])
bound = np.prod(np.linalg.norm(Hm, axis=0))
print(f"|det| = {abs(np.linalg.det(Hm)):.6f}   Hadamard bound = {bound:.6f}")

Qo, _ = np.linalg.qr(rng.standard_normal((4, 4)))
Sc = Qo @ np.diag([2.0, 3.0, 0.5, 1.5])
print(f"orthogonal columns: |det| = {abs(np.linalg.det(Sc)):.10f}   bound = "
      f"{np.prod(np.linalg.norm(Sc, axis=0)):.10f}")

worst = 0.0
for _ in range(2000):
    Rm = rng.standard_normal((4, 4))
    worst = max(worst, abs(np.linalg.det(Rm)) / np.prod(np.linalg.norm(Rm, axis=0)))
print(f"largest ratio |det| / bound over 2000 random 4x4 matrices: {worst:.6f}  (must not exceed 1)")
assert abs(np.linalg.det(Hm)) <= bound + 1e-12
assert worst <= 1.0 + 1e-12

|det| = 1.000000   Hadamard bound = 43.954522
orthogonal columns: |det| = 4.5000000000   bound = 4.5000000000
largest ratio |det| / bound over 2000 random 4x4 matrices: 0.959615  (must not exceed 1)


### Problem L3.6 — The Cauchy-Binet formula

**Statement.** For $A \in \mathbb{F}^{m \times n}$ and $B \in \mathbb{F}^{n \times m}$ with
$m \le n$, prove

$$
\det(AB) = \sum_{S} \det\bigl( A_{[m], S} \bigr) \det\bigl( B_{S, [m]} \bigr),
$$

the sum running over the $\binom{n}{m}$ subsets $S \subseteq \{1, \dots, n\}$ of size $m$, and
show $\det(AB) = 0$ when $m \gt n$.

**Intuition.** Expand the determinant of $AB$ multilinearly in its columns; every non-vanishing
term picks $m$ distinct columns of $A$.

**Solution.**

*Step 1 — expand the columns.* The $j$-th column of $AB$ is $\sum_{k=1}^{n} B_{kj} a_k$, where
$a_k$ is the $k$-th column of $A$. Multilinearity gives

$$
\det(AB) = \sum_{f : [m] \to [n]} \left( \prod_{j=1}^{m} B_{f(j), j} \right) \det\bigl( a_{f(1)}, \dots, a_{f(m)} \bigr).
$$

*Step 2 — discard the non-injective terms.* If $f$ repeats a value the determinant has two equal
columns and vanishes. Only injections survive; if $m \gt n$ there are none, which proves the
second claim.

*Step 3 — group the injections.* An injection is determined by its image
$S = \{k_1 \lt \cdots \lt k_m\}$ together with a permutation $\sigma \in S_m$ via
$f(j) = k_{\sigma(j)}$. Reordering the columns costs the sign,

$$
\det\bigl( a_{k_{\sigma(1)}}, \dots, a_{k_{\sigma(m)}} \bigr) = \operatorname{sgn}(\sigma) \det\bigl( A_{[m], S} \bigr).
$$

*Step 4 — recognize the inner sum.* For fixed $S$,

$$
\sum_{\sigma \in S_m} \operatorname{sgn}(\sigma) \prod_{j=1}^{m} B_{k_{\sigma(j)}, j} = \det\bigl( B_{S, [m]} \bigr)
$$

is exactly the Leibniz formula of Theorem 4.1 for the $m \times m$ matrix whose $(i,j)$ entry is
$B_{k_i, j}$.

$$
\boxed{\det(AB) = \sum_{\lvert S \rvert = m} \det\bigl(A_{[m],S}\bigr)\det\bigl(B_{S,[m]}\bigr)}
$$

**Key takeaway.** With $m = n$ there is a single subset and the formula collapses to
Theorem 4.3. With $B = A^{\top}$ it becomes $\det(AA^{\top}) = \sum_S \det(A_{[m],S})^{2}$,
which is the engine of the Matrix-Tree theorem in Problem L3.10.

In [48]:
m, n = 3, 5
A = rng.integers(-3, 4, size=(m, n)).astype(float)
B = rng.integers(-3, 4, size=(n, m)).astype(float)
total = sum(np.linalg.det(A[:, S]) * np.linalg.det(B[S, :])
            for S in itertools.combinations(range(n), m))
print(f"det(AB)          = {np.linalg.det(A @ B):+.10f}")
print(f"Cauchy-Binet sum = {total:+.10f}   over {len(list(itertools.combinations(range(n), m)))} subsets")
gram = sum(np.linalg.det(A[:, S]) ** 2 for S in itertools.combinations(range(n), m))
print(f"det(A A^T)       = {np.linalg.det(A @ A.T):+.10f}   sum of squared minors = {gram:+.10f}")
tall = rng.standard_normal((4, 2))
print(f"m > n case: det of a 4x4 product of rank 2 = {np.linalg.det(tall @ tall.T):.3e}")
assert abs(np.linalg.det(A @ B) - total) < 1e-8
assert abs(np.linalg.det(A @ A.T) - gram) < 1e-8
assert abs(np.linalg.det(tall @ tall.T)) < 1e-10

det(AB)          = -162.0000000000
Cauchy-Binet sum = -162.0000000000   over 10 subsets
det(A A^T)       = +507.0000000000   sum of squared minors = +507.0000000000
m > n case: det of a 4x4 product of rank 2 = 8.549e-31


### Problem L3.7 — The Vandermonde determinant in general

**Statement.** Prove that for the $n \times n$ matrix $V$ with $V_{ij} = x_j^{\,i-1}$,

$$
\det V = \prod_{1 \le i \lt j \le n} (x_j - x_i).
$$

**Intuition.** As a polynomial in $x_n$ the determinant has known roots and known degree, which
pins it down up to the leading coefficient.

**Solution.**

*Step 1 — induction setup.* The claim holds for $n = 1$ (empty product, $\det V = 1$). Assume it
for $n - 1$.

*Step 2 — clear the first column.* Perform the row operations
$R_i \mapsto R_i - x_1 R_{i-1}$ for $i = n, n-1, \dots, 2$, in that order. Each adds a multiple
of another row, so by Theorem 4.2 and multilinearity none changes the determinant.

*Step 3 — read the new matrix.* The $(i,j)$ entry becomes
$x_j^{\,i-1} - x_1 x_j^{\,i-2} = x_j^{\,i-2}(x_j - x_1)$ for $i \ge 2$, while the first row is
unchanged. Column $1$ becomes $(1, 0, \dots, 0)^{\top}$.

*Step 4 — expand and factor.* Laplace expansion along the first column (Theorem 4.4) leaves the
$(n-1) \times (n-1)$ determinant of the matrix with entries $x_j^{\,i-1}(x_j - x_1)$ for
$j = 2, \dots, n$. Pulling $(x_j - x_1)$ out of column $j$,

$$
\det V = \left( \prod_{j=2}^{n} (x_j - x_1) \right) \det V',
$$

where $V'$ is the Vandermonde matrix on $x_2, \dots, x_n$.

*Step 5 — apply the hypothesis.* $\det V' = \prod_{2 \le i \lt j \le n}(x_j - x_i)$, and
combining the two products gives the claim.

$$
\boxed{\det V = \prod_{1 \le i \lt j \le n} (x_j - x_i)}
$$

**Key takeaway.** $\det V \neq 0$ exactly when the nodes are distinct. That is the existence and
uniqueness theorem for polynomial interpolation, and the determinant's rapid decay for clustered
nodes is why interpolation at nearly equal nodes is ill conditioned.

In [49]:
for xs in [np.array([1.0, 2.0, 4.0, 8.0]), rng.standard_normal(5), np.linspace(0.0, 1.0, 6)]:
    n = len(xs)
    V = np.vander(xs, increasing=True).T
    formula = prod(xs[j] - xs[i] for i in range(n) for j in range(i + 1, n))
    print(f"n = {n}:  det V = {np.linalg.det(V):+.8e}   product formula = {formula:+.8e}"
          f"   relative diff {abs(np.linalg.det(V) - formula) / max(abs(formula), 1e-30):.2e}")
    assert abs(np.linalg.det(V) - formula) < 1e-8 * max(abs(formula), 1.0)

n = 4:  det V = +1.00800000e+03   product formula = +1.00800000e+03   relative diff 6.77e-16
n = 5:  det V = -2.33399874e-02   product formula = -2.33399874e-02   relative diff 2.97e-16
n = 6:  det V = +1.13246208e-06   product formula = +1.13246208e-06   relative diff 2.11e-14


### Problem L3.8 — Adjugate identities

**Statement.** For $A \in \mathbb{F}^{n \times n}$ with $n \ge 2$, prove

$$
\operatorname{rank}(\operatorname{adj}A) = \begin{cases} n & \operatorname{rank}A = n \\ 1 & \operatorname{rank}A = n-1 \\ 0 & \operatorname{rank}A \le n-2 \end{cases}
\qquad \text{and} \qquad
\operatorname{adj}(\operatorname{adj}A) = (\det A)^{\,n-2} A .
$$

**Intuition.** The adjugate is built from $(n-1)$-minors, so it detects exactly how far $A$ is
from having rank $n-1$.

**Solution.**

*Step 1 — full rank.* If $\det A \neq 0$ then $\operatorname{adj}A = \det(A)A^{-1}$ is
invertible, so its rank is $n$.

*Step 2 — rank at most $n-2$.* Every $(n-1) \times (n-1)$ submatrix of $A$ has rank at most
$\operatorname{rank}A \le n - 2 \lt n-1$, so every cofactor vanishes and
$\operatorname{adj}A = 0$.

*Step 3 — rank exactly $n-1$.* Some $(n-1)$-minor is non-zero, so $\operatorname{adj}A \neq 0$
and the rank is at least $1$. Theorem 4.4 gives $A \operatorname{adj}(A) = \det(A) I = 0$, so
every column of $\operatorname{adj}A$ lies in $\operatorname{Null}(A)$, which has dimension
$n - (n-1) = 1$. Hence the rank is exactly $1$.

*Step 4 — the double adjugate, invertible case.* From $\operatorname{adj}A = \det(A)A^{-1}$ and
Theorem 4.3,

$$
\det(\operatorname{adj}A) = \det(A)^{n} \det(A^{-1}) = \det(A)^{n-1},
\qquad
(\operatorname{adj}A)^{-1} = \frac{A}{\det A}.
$$

Applying $\operatorname{adj}M = \det(M)M^{-1}$ to $M = \operatorname{adj}A$,

$$
\operatorname{adj}(\operatorname{adj}A) = \det(A)^{n-1} \cdot \frac{A}{\det A} = (\det A)^{n-2} A .
$$

*Step 5 — the singular case.* If $\det A = 0$ and $n \ge 3$, the right-hand side is $0$. For the
left-hand side: if $\operatorname{rank}A \le n-2$ then $\operatorname{adj}A = 0$ by Step 2 and
its adjugate is $0$; if $\operatorname{rank}A = n-1$ then $\operatorname{adj}A$ has rank $1$ by
Step 3, so all its $(n-1)$-minors vanish because $n - 1 \ge 2$, giving
$\operatorname{adj}(\operatorname{adj}A) = 0$ again. For $n = 2$ the identity reads
$\operatorname{adj}(\operatorname{adj}A) = A$, which holds for every $A$ by direct computation.

$$
\boxed{\operatorname{adj}(\operatorname{adj}A) = (\det A)^{\,n-2} A}
$$

**Key takeaway.** The rank trichotomy is the reason $\operatorname{adj}A$ is the standard tool for
producing a null vector of a matrix known to have a one-dimensional kernel — any non-zero column
of the adjugate will do.

In [50]:
def adjugate(M):
    """Adjugate by cofactors, valid for singular M as well."""
    n = M.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            C[i, j] = (-1) ** (i + j) * np.linalg.det(np.delete(np.delete(M, i, 0), j, 1))
    return C.T

n = 4
full = rng.standard_normal((n, n))
rank3 = np.column_stack([rng.standard_normal((n, 3)), np.zeros(n)])
rank3 = rank3 @ rng.standard_normal((n, n))
rank2 = rng.standard_normal((n, 2)) @ rng.standard_normal((2, n))
for M, name in [(full, "rank 4"), (rank3, "rank 3"), (rank2, "rank 2")]:
    ad = adjugate(M)
    print(f"{name}: rank(A) = {np.linalg.matrix_rank(M)}   rank(adj A) = "
          f"{np.linalg.matrix_rank(ad, tol=1e-8)}   ||adj A||_F = {np.linalg.norm(ad):.3e}")
print("\ndouble adjugate on the invertible matrix:")
lhs = adjugate(adjugate(full))
rhs = np.linalg.det(full) ** (n - 2) * full
print(f"  relative difference = {np.linalg.norm(lhs - rhs) / np.linalg.norm(rhs):.3e}")
assert np.linalg.matrix_rank(adjugate(rank3), tol=1e-8) == 1
assert np.linalg.norm(adjugate(rank2)) < 1e-8
assert np.linalg.norm(lhs - rhs) / np.linalg.norm(rhs) < 1e-9

rank 4: rank(A) = 4   rank(adj A) = 4   ||adj A||_F = 9.317e+00
rank 3: rank(A) = 3   rank(adj A) = 1   ||adj A||_F = 9.989e-01
rank 2: rank(A) = 2   rank(adj A) = 0   ||adj A||_F = 3.926e-16

double adjugate on the invertible matrix:
  relative difference = 3.768e-16


### Problem L3.9 — Minimal polynomial of an explicit $4 \times 4$

**Statement.** Compute the characteristic and minimal polynomials of

$$
J = \begin{pmatrix}
3 & 1 & 0 & 0 \\
0 & 3 & 1 & 0 \\
0 & 0 & 3 & 0 \\
0 & 0 & 0 & 3
\end{pmatrix}
$$

and deduce that $J$ is not diagonalizable.

**Intuition.** The largest chain of off-diagonal ones fixes the exponent in the minimal
polynomial.

**Solution.**

*Step 1 — characteristic polynomial.* $J$ is triangular, so by Problem L0.2
$p_J(t) = (t-3)^{4}$, and $3$ is the only eigenvalue with algebraic multiplicity $4$.

*Step 2 — powers of the nilpotent part.* Put $N = J - 3I$, which has ones in positions
$(1,2)$ and $(2,3)$ and zeros elsewhere. Then

$$
N^{2} = \begin{pmatrix} 0&0&1&0 \\ 0&0&0&0 \\ 0&0&0&0 \\ 0&0&0&0 \end{pmatrix} \neq 0,
\qquad
N^{3} = 0 .
$$

*Step 3 — read off the minimal polynomial.* Theorem 4.8 says $m_J$ divides $p_J$, so
$m_J(t) = (t-3)^{d}$ for some $d \le 4$. Step 2 gives $d = 3$.

*Step 4 — diagonalizability.* $\operatorname{rank}(N) = 2$, so
$\dim\operatorname{Null}(J - 3I) = 2 \lt 4$: there are only two independent eigenvectors, and
$J$ has no eigenbasis. Equivalently, a matrix is diagonalizable if and only if its minimal
polynomial has only simple roots, and $(t-3)^{3}$ does not.

$$
\boxed{p_J(t) = (t-3)^{4}, \qquad m_J(t) = (t-3)^{3}, \qquad J \text{ is not diagonalizable}}
$$

**Key takeaway.** $\deg m_J = 3$ is the size of the largest Jordan block and
$4 - \operatorname{rank}(N) = 2$ is the number of blocks. Both numbers are read off powers of
$J - 3I$, with no eigenvector computation.

In [51]:
J = np.array([[3.0, 1.0, 0.0, 0.0],
              [0.0, 3.0, 1.0, 0.0],
              [0.0, 0.0, 3.0, 0.0],
              [0.0, 0.0, 0.0, 3.0]])
N = J - 3 * np.eye(4)
print("characteristic polynomial:", np.round(np.poly(J), 10), " -> (t-3)^4 =",
      np.round(np.poly(3 * np.ones(4)), 10))
for k in (1, 2, 3):
    print(f"  ||N^{k}||_F = {np.linalg.norm(np.linalg.matrix_power(N, k)):.3f}")
print("  dim Null(J - 3I) =", 4 - np.linalg.matrix_rank(N))
print("  number of independent eigenvectors:", 4 - np.linalg.matrix_rank(N), "< 4, so not diagonalizable")
assert np.linalg.norm(np.linalg.matrix_power(N, 2)) > 0.5
assert np.linalg.norm(np.linalg.matrix_power(N, 3)) < 1e-12
assert 4 - np.linalg.matrix_rank(N) == 2

characteristic polynomial: [   1.  -12.   54. -108.   81.]  -> (t-3)^4 = [   1.  -12.   54. -108.   81.]
  ||N^1||_F = 1.414
  ||N^2||_F = 1.000
  ||N^3||_F = 0.000
  dim Null(J - 3I) = 2
  number of independent eigenvectors: 2 < 4, so not diagonalizable


### Problem L3.10 — The Matrix-Tree theorem

**Statement.** Let $G$ be a graph on $n$ vertices with Laplacian $L = D - W$. Prove that the
principal cofactor obtained by deleting row and column $r$ equals the number of spanning trees
of $G$, for every $r$.

**Intuition.** Cauchy-Binet turns $\det(N_0 N_0^{\top})$ into a sum over edge subsets, and each
subset contributes $1$ exactly when it is a spanning tree.

**Solution.**

*Step 1 — incidence matrix.* Orient every edge arbitrarily and let
$N \in \{0, \pm 1\}^{n \times m}$ have $N_{ve} = +1$ if $v$ is the head of $e$, $-1$ if $v$ is
the tail, and $0$ otherwise. Then

$$
(N N^{\top})_{uv} = \sum_{e} N_{ue}N_{ve} =
\begin{cases}
\deg(u) & u = v \\
-1 & uv \text{ an edge} \\
0 & \text{otherwise},
\end{cases}
$$

so $L = N N^{\top}$.

*Step 2 — the cofactor is a Gram determinant.* Let $N_0 \in \mathbb{F}^{(n-1) \times m}$ be $N$
with row $r$ deleted. Deleting row and column $r$ from $L = NN^{\top}$ gives exactly
$N_0 N_0^{\top}$, so the cofactor equals $\det(N_0 N_0^{\top})$.

*Step 3 — Cauchy-Binet.* By Problem L3.6 with $A = N_0$ and $B = N_0^{\top}$,

$$
\det\bigl( N_0 N_0^{\top} \bigr) = \sum_{\lvert S \rvert = n-1} \det\bigl( N_{0, S} \bigr)^{2},
$$

the sum running over sets $S$ of $n-1$ edges.

*Step 4 — the key lemma.* $\det(N_{0,S}) = \pm 1$ if $S$ is a spanning tree and $0$ otherwise.

If $S$ contains a cycle, the signed sum of the corresponding columns of $N$ around that cycle is
zero, so the columns are dependent and the determinant vanishes. Since $\lvert S \rvert = n-1$, a
subgraph is a spanning tree exactly when it is acyclic.

If $S$ is a spanning tree, induct on $n$. A tree on at least two vertices has a leaf $v \neq r$;
the row of $v$ in $N_{0,S}$ has a single non-zero entry $\pm 1$, in the column of its unique
incident edge. Expanding along that row (Theorem 4.4) multiplies $\pm 1$ by the corresponding
minor, which is the same construction for the tree with $v$ and its edge removed. The induction
terminates at the $1 \times 1$ case with value $\pm 1$.

*Step 5 — conclude.* Every spanning tree contributes $(\pm 1)^{2} = 1$ and every other subset
contributes $0$, so the cofactor counts spanning trees. The value does not depend on $r$, because
the count does not.

$$
\boxed{\det\bigl( L_{\hat r, \hat r} \bigr) = \#\{\text{spanning trees of } G\} \quad \text{for every } r}
$$

**Key takeaway.** A determinant computed in $\mathcal{O}(n^{3})$ replaces an enumeration over
exponentially many edge subsets. For the complete graph $K_n$ the cofactor evaluates to
$n^{n-2}$, which is Cayley's formula.

In [52]:
def spanning_trees_bruteforce(n, edges):
    count = 0
    for sub in itertools.combinations(edges, n - 1):
        Ms = np.zeros((n, n))
        for a, b in sub:
            Ms[a, b] = Ms[b, a] = 1.0
        if np.linalg.matrix_rank(np.diag(Ms.sum(axis=1)) - Ms) == n - 1:
            count += 1
    return count

def laplacian(n, edges):
    Wg = np.zeros((n, n))
    for a, b in edges:
        Wg[a, b] = Wg[b, a] = 1.0
    return np.diag(Wg.sum(axis=1)) - Wg

tests = [
    ("path on 4", 4, [(0, 1), (1, 2), (2, 3)], False),
    ("cycle on 4", 4, [(0, 1), (1, 2), (2, 3), (3, 0)], False),
    ("K4", 4, list(itertools.combinations(range(4), 2)), True),
    ("K5", 5, list(itertools.combinations(range(5), 2)), True),
]
for name, n, edges, complete in tests:
    L = laplacian(n, edges)
    cofs = [round(np.linalg.det(np.delete(np.delete(L, r, 0), r, 1))) for r in range(n)]
    brute = spanning_trees_bruteforce(n, edges)
    tail = f"   Cayley n^(n-2) = {n ** (n - 2)}" if complete else ""
    print(f"{name:<12s} cofactors {cofs}   brute force {brute}{tail}")
    assert cofs == [brute] * n
    if complete:
        assert brute == n ** (n - 2)
assert round(np.linalg.det(np.delete(np.delete(laplacian(5, list(itertools.combinations(range(5), 2))), 0, 0), 0, 1))) == 125

path on 4    cofactors [1, 1, 1, 1]   brute force 1
cycle on 4   cofactors [4, 4, 4, 4]   brute force 4
K4           cofactors [16, 16, 16, 16]   brute force 16   Cayley n^(n-2) = 16
K5           cofactors [125, 125, 125, 125, 125]   brute force 125   Cayley n^(n-2) = 125


### Problem L3.11 — Newton's identities and Faddeev-LeVerrier

**Statement.** With $E_k$ the coefficients of Theorem 4.6 and $p_k = \operatorname{tr}(A^{k})$,
prove Newton's identities

$$
p_k = E_1 p_{k-1} - E_2 p_{k-2} + \cdots + (-1)^{k-2} E_{k-1} p_1 + (-1)^{k-1} k E_k,
\qquad 1 \le k \le n,
$$

and explain why they let the characteristic polynomial be computed from traces alone.

**Intuition.** Take the logarithmic derivative of the factored characteristic polynomial.

**Solution.**

*Step 1 — generating function.* Over $\mathbb{C}$ write $\lambda_1, \dots, \lambda_n$ for the
roots of $p_A$. By Theorem 4.6, $E_k = e_k(\lambda)$, and $p_k = \sum_i \lambda_i^{k}$. Set

$$
E(x) = \prod_{i=1}^{n} (1 - \lambda_i x) = \sum_{k=0}^{n} (-1)^{k} E_k x^{k} .
$$

*Step 2 — logarithmic derivative.*

$$
\frac{E'(x)}{E(x)} = \sum_{i=1}^{n} \frac{-\lambda_i}{1 - \lambda_i x} = -\sum_{k \ge 1} p_k x^{k-1},
$$

expanding each term as a geometric series for small $\lvert x \rvert$.

*Step 3 — clear the denominator.* Hence
$E'(x) = -E(x)\sum_{k \ge 1} p_k x^{k-1}$ as formal power series.

*Step 4 — compare coefficients of $x^{k-1}$.* On the left the coefficient is
$k(-1)^{k}E_k$; on the right it is $-\sum_{j=0}^{k-1} (-1)^{j} E_j \, p_{k-j}$. Equating and
multiplying by $(-1)^{k-1}$ gives the stated identity.

*Step 5 — the algorithmic consequence.* The identities determine $E_1, \dots, E_n$ recursively
from $p_1, \dots, p_n$. Faddeev-LeVerrier is the matrix form of that recursion: starting from
$M_0 = 0$ and $c_0 = 1$, iterate

$$
M_k = A M_{k-1} + c_{k-1} I, \qquad c_k = -\frac{1}{k}\operatorname{tr}(A M_k),
$$

producing $p_A(t) = \sum_k c_k t^{n-k}$ using only matrix products and traces.

$$
\boxed{p_k = \sum_{j=1}^{k-1} (-1)^{j-1} E_j \, p_{k-j} + (-1)^{k-1} k E_k}
$$

**Key takeaway.** No minor is ever formed. The characteristic polynomial costs $n$ matrix
multiplications, which is why Faddeev-LeVerrier remains the textbook symbolic algorithm even
though it is numerically fragile for large $n$.

In [53]:
n = 5
A = rng.standard_normal((n, n))
coeffs = np.poly(A)
E = np.array([1.0] + [(-1) ** k * coeffs[k] for k in range(1, n + 1)])
pk = np.array([np.trace(np.linalg.matrix_power(A, k)) for k in range(1, n + 1)])
print("E_k :", E)
print("p_k :", pk)
for k in range(1, n + 1):
    rhs = sum((-1) ** (j - 1) * E[j] * pk[k - j - 1] for j in range(1, k)) + (-1) ** (k - 1) * k * E[k]
    print(f"  k = {k}:  p_k = {pk[k - 1]:+.10f}   Newton right-hand side = {rhs:+.10f}")
    assert abs(pk[k - 1] - rhs) < 1e-8

def faddeev_leverrier(M):
    d = M.shape[0]
    Mk = np.zeros_like(M)
    c = np.zeros(d + 1)
    c[0] = 1.0
    for k in range(1, d + 1):
        Mk = M @ Mk + c[k - 1] * np.eye(d)
        c[k] = -np.trace(M @ Mk) / k
    return c

print("\nFaddeev-LeVerrier:", faddeev_leverrier(A))
print("numpy.poly       :", coeffs)
assert np.abs(faddeev_leverrier(A) - coeffs).max() < 1e-9

E_k : [ 1.     -1.0985 -2.7098  0.6743  3.6922 -3.117 ]
p_k : [ -1.0985   6.6263  -8.2327  11.4902 -41.992 ]
  k = 1:  p_k = -1.0984718920   Newton right-hand side = -1.0984718920
  k = 2:  p_k = +6.6263204348   Newton right-hand side = +6.6263204348
  k = 3:  p_k = -8.2326838179   Newton right-hand side = -8.2326838179
  k = 4:  p_k = +11.4902254114   Newton right-hand side = +11.4902254114
  k = 5:  p_k = -41.9919868793   Newton right-hand side = -41.9919868793

Faddeev-LeVerrier: [ 1.      1.0985 -2.7098 -0.6743  3.6922  3.117 ]
numpy.poly       : [ 1.      1.0985 -2.7098 -0.6743  3.6922  3.117 ]


### Problem L3.12 — $-\log\det$ is convex on the positive definite cone

**Statement.** Prove that $f(X) = -\log\det X$ is convex on
$\{ X \in \mathbb{R}^{n \times n} : X = X^{\top}, \ X \succ 0 \}$.

**Intuition.** Restrict to a line and show the second derivative is a squared norm.

**Solution.**

*Step 1 — reduce to a line.* A function is convex on a convex set exactly when its restriction to
every line segment inside the set is convex. Fix $X \succ 0$ and a symmetric $H$, and let
$g(t) = \log\det(X + tH)$ on the interval where $M(t) = X + tH \succ 0$; that set is an interval
because the definite cone is convex.

*Step 2 — second derivative.* By Problem L3.1,

$$
g''(t) = -\operatorname{tr}\bigl( M^{-1} H M^{-1} H \bigr), \qquad M = M(t) .
$$

*Step 3 — factor the inverse.* $M \succ 0$ implies $M^{-1} \succ 0$, so the Cholesky
factorization of [Module 03](../03_linear_systems_and_direct_factorizations/) gives
$M^{-1} = L L^{\top}$ with $L$ invertible.

*Step 4 — recognize a squared norm.* Using cyclic invariance (Theorem 4.5),

$$
\operatorname{tr}\bigl( LL^{\top} H LL^{\top} H \bigr) = \operatorname{tr}\bigl( (L^{\top}HL)(L^{\top}HL) \bigr) = \lVert L^{\top}HL \rVert_F^{2} \ \ge \ 0,
$$

the last equality because $S = L^{\top}HL$ is symmetric, so
$\operatorname{tr}(S^{2}) = \operatorname{tr}(S^{\top}S) = \lVert S \rVert_F^{2}$.

*Step 5 — conclude.* Hence $g'' \le 0$, so $g$ is concave and $f = -g$ is convex. Strictness:
$\lVert L^{\top}HL \rVert_F = 0$ forces $H = 0$ because $L$ is invertible, so $f$ is strictly
convex.

$$
\boxed{-\log\det X \text{ is strictly convex on } \{X = X^{\top} \succ 0\}}
$$

**Key takeaway.** This single fact makes maximum-likelihood covariance estimation, the graphical
lasso, D-optimal experiment design and every log-determinant barrier in interior-point methods
convex problems.

In [54]:
n = 4
def rand_pd(d):
    Z = rng.standard_normal((d, d))
    return Z @ Z.T + d * np.eye(d)

X1, X2 = rand_pd(n), rand_pd(n)
print("midpoint test on 5 random pairs (f((X+Y)/2) <= (f(X)+f(Y))/2):")
for trial in range(5):
    X1, X2 = rand_pd(n), rand_pd(n)
    f = lambda M: -np.log(np.linalg.det(M))
    mid, avg = f((X1 + X2) / 2), (f(X1) + f(X2)) / 2
    print(f"  trial {trial}: f(mid) = {mid:+.6f} <= {avg:+.6f} = average    gap {avg - mid:.6f}")
    assert mid <= avg + 1e-10

H = rng.standard_normal((n, n))
H = (H + H.T) / 2
Xi = np.linalg.inv(X1)
Lc = np.linalg.cholesky(Xi)
second = -np.trace(Xi @ H @ Xi @ H)
print(f"\ng''(0)                 = {second:.8f}")
print(f"-||L^T H L||_F^2       = {-np.linalg.norm(Lc.T @ H @ Lc) ** 2:.8f}")
assert second <= 0
assert abs(second + np.linalg.norm(Lc.T @ H @ Lc) ** 2) < 1e-8

midpoint test on 5 random pairs (f((X+Y)/2) <= (f(X)+f(Y))/2):
  trial 0: f(mid) = -7.913047 <= -7.596182 = average    gap 0.316865
  trial 1: f(mid) = -8.402463 <= -8.214937 = average    gap 0.187525
  trial 2: f(mid) = -7.934818 <= -7.693790 = average    gap 0.241029
  trial 3: f(mid) = -8.194343 <= -8.000591 = average    gap 0.193752
  trial 4: f(mid) = -7.902697 <= -7.685052 = average    gap 0.217645

g''(0)                 = -0.09425557
-||L^T H L||_F^2       = -0.09425557


### Problem L3.13 — The equation $ST - TS = I$ has no solution

**Statement.** Prove that no matrices $S, T \in \mathbb{F}^{n \times n}$ satisfy $ST - TS = I$
when $\mathbb{F}$ has characteristic zero, and show that the statement genuinely needs that
hypothesis.

**Intuition.** Take traces: the left side is always $0$ and the right side is $n$.

**Solution.**

*Step 1 — the trace of the left side.* By Problem L1.1,
$\operatorname{tr}(ST - TS) = 0$ for any $S, T$.

*Step 2 — the trace of the right side.* $\operatorname{tr}(I_n) = n$ by Problem L0.1.

*Step 3 — conclude.* If $ST - TS = I$ then $0 = n$ in $\mathbb{F}$, impossible in characteristic
zero.

*Step 4 — the hypothesis is real.* Over $\mathbb{F}_2$ with $n = 2$ the scalar $n$ *is* zero, and
a solution exists:

$$
S = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}, \quad
T = \begin{pmatrix} 0 & 0 \\ 1 & 0 \end{pmatrix},
\quad
ST - TS = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \equiv \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} \pmod 2 .
$$

$$
\boxed{ST - TS = I \text{ is unsolvable in } \mathbb{F}^{n \times n} \text{ when } \operatorname{char}\mathbb{F} = 0}
$$

**Key takeaway.** The canonical commutation relation $[\hat{x}, \hat{p}] = i\hbar I$ of quantum
mechanics therefore has no finite-dimensional representation: position and momentum are forced
to be unbounded operators on an infinite-dimensional space.

In [55]:
print("real matrices: the commutator always has zero trace")
worst = 0.0
for _ in range(2000):
    S = rng.standard_normal((3, 3))
    T = rng.standard_normal((3, 3))
    worst = max(worst, abs(np.trace(S @ T - T @ S)))
print(f"  largest |tr(ST - TS)| over 2000 random pairs = {worst:.3e}")
print(f"  tr(I_3) = {np.trace(np.eye(3)):.0f}, so ST - TS = I is impossible")

S2 = np.array([[0, 1], [0, 0]])
T2 = np.array([[0, 0], [1, 0]])
comm = S2 @ T2 - T2 @ S2
print("\nover F_2 with n = 2:")
print("  ST - TS over the integers:\n", comm)
print("  reduced mod 2:\n", comm % 2)
assert worst < 1e-12
assert np.array_equal(comm % 2, np.eye(2, dtype=int))

real matrices: the commutator always has zero trace
  largest |tr(ST - TS)| over 2000 random pairs = 2.137e-15
  tr(I_3) = 3, so ST - TS = I is impossible

over F_2 with n = 2:
  ST - TS over the integers:
 [[ 1  0]
 [ 0 -1]]
  reduced mod 2:
 [[1 0]
 [0 1]]
